# 02 · RFM-proxy theo tháng | Santander Track DS, Checkpoint 1 (build lại, Đợt 1–3)

**Nguồn chuẩn cho R/F/M:** hai output của `01-rfm-proxy-events.ipynb` (`acquisition_events.parquet`, `rfm_proxy_raw.parquet`). Notebook này đọc thêm `train_ver2.csv.zip` để lấy snapshot từng tháng, tự tạo cửa sổ quan sát (không cần `customer_observation_window.parquet`).

| Đợt | Bước | Trạng thái |
|---|---|---|
| 1 | Chuẩn hóa đầu vào + feature khách × tháng + đối soát với notebook 01 | xong (mục 0–3) |
| **2** | Bốn cấu hình RFM (`F_events`/`F_event_months` × `M_n_products`/`M_renta`) × quantile / k-means / rule-based | **notebook này** (mục 4–6) |
| 3 | Kiểm chứng dự báo: lift@10%, PR-AUC, nhãn sản phẩm mới | xong (mục 7) |

**Định nghĩa (khớp notebook 01):** `F_events` = số event 0→1; `F_event_months` = số tháng có event (**F chính**); `R` = tháng chốt − tháng của event gần nhất (NaN nếu chưa có event); `M_n_products` = tổng 24 cờ tại snapshot của tháng (NaN nếu thiếu bất kỳ cờ); `M_renta` = renta hợp lệ gần nhất đến tháng đó, **không điền**.

**Khách vắng tháng đó:** vẫn có dòng (`is_observed=False`, `is_stale=True`); các giá trị danh mục/active/renta là giá trị cũ từ lần thấy gần nhất, không phải sở hữu hiện tại.

| Output Đợt 1 | Nội dung |
|---|---|
| `customer_month_features/customer_features_YYYY-MM.parquet` | 16 cột: trạng thái quan sát, R/F/M, active (mang từ lần thấy gần nhất), `observed_months`, `tenure_months` |
| `rfm_labels/rfm_labels_<cấu hình>_YYYY-MM.parquet` | Đợt 2: nhãn quantile / k-means / rule-based của từng cấu hình tại các tháng chốt |
| `rfm_thresholds.json` | Đợt 2: ngưỡng quantile, tâm cụm k-means, ngưỡng rule-based, giá trị điền income, phiên bản phương pháp |


## 📖 Đọc trước: notebook này làm gì? (giải thích cho người mới)

### 1. Bài toán
Ngân hàng Santander có khoảng 950 nghìn khách hàng. Mỗi tháng từ 01/2015 đến 05/2016 (17 tháng), dữ liệu có một "ảnh chụp" (*snapshot*) cho biết khách đang giữ sản phẩm nào trong **24 sản phẩm** (tài khoản vãng lai, thẻ tín dụng, quỹ đầu tư, khoản vay...).

**Câu hỏi:** làm sao chia khách thành vài nhóm có ý nghĩa, ví dụ "khách gắn bó", "khách sắp rời đi", để ngân hàng biết nên chăm sóc ai, bán thêm sản phẩm gì cho ai? Cách chia kinh điển là **RFM**. Notebook này dựng RFM từ dữ liệu, chia nhóm bằng ba cách, rồi kiểm tra xem nhóm nào thật sự hữu ích.

### 2. RFM là gì, và áp vào dữ liệu này thế nào
RFM thường dùng trong bán lẻ: **R**ecency (lần mua gần nhất cách đây bao lâu), **F**requency (mua bao nhiêu lần), **M**onetary (tiêu bao nhiêu tiền). Dữ liệu ngân hàng này **không có số tiền giao dịch**, nên notebook dùng đại lượng thay thế (*proxy*) dựa trên **event** = *khách mở thêm một sản phẩm* (cờ sản phẩm chuyển từ 0 sang 1 giữa hai tháng liền kề):

| Chữ | Nghĩa trong notebook | Càng tốt khi |
|---|---|---|
| **R** | số tháng kể từ lần mở sản phẩm gần nhất | càng nhỏ (mới mở gần đây) |
| **F** | số tháng khách có mở sản phẩm (`F_event_months`) | càng lớn |
| **M** | số sản phẩm khách đang giữ (`M_n_products`) | càng lớn |

**Ví dụ:** khách A mở thẻ tín dụng 1 tháng trước, trong 17 tháng có 3 tháng mở sản phẩm, đang giữ 4 sản phẩm → R=1, F=3, M=4 (gắn bó). Khách B mở lần cuối cách đây 10 tháng, chỉ mở 1 lần, giữ 2 sản phẩm → R=10, F=1, M=2 (ít gắn bó).

**Lưu ý:** 78,7% khách chưa từng mở thêm sản phẩm nào trong cửa sổ quan sát, nên không có R. Họ được tách riêng thành nhóm **`No_Acquisition_Event`**, không bị coi là "xấu" hay "đã rời đi".

### 3. Notebook làm gì, theo từng mục
| Mục | Việc làm |
|---|---|
| 0–1 | Cấu hình và các hàm đọc dữ liệu thô (CSV khoảng 13,65 triệu dòng) |
| 2 | Đọc dữ liệu thật, kiểm tra các file đầu vào cùng một phiên bản |
| 3 | Tính R, F, M cho **từng khách ở từng tháng**; kiểm tra khớp với notebook 01 và "không nhìn tương lai" |
| 4–5 | Chia khách thành nhóm bằng **3 cách** (quantile, k-means, rule-based) × **4 cách đo RFM**; học ngưỡng ở 08/2015 rồi áp lên các tháng sau |
| 6 | So sánh: bao nhiêu khách được chấm, nhóm có ổn định theo tháng không, khách nào đổi nhóm khi đổi cách đo |
| 7 | **Kiểm chứng:** nhóm hôm nay có đoán được ai sẽ mở sản phẩm trong 3 tháng sau không |
| 8 | Kết luận: chọn cấu hình nào và vì sao |
| 9 | Hướng mở rộng |

### 4. Ba cách chia nhóm (giải thích đời thường)
- **Quantile:** xếp khách theo từng chỉ số rồi chia thành 3 mức (thấp / vừa / cao), như xếp học sinh giỏi / khá / trung bình. Ghép ba chỉ số lại ra 27 nhóm, ví dụ **R3F3M1**. Số lớn là tốt: R3 = mở gần đây nhất, F3 = mở nhiều nhất, M3 = giữ nhiều sản phẩm nhất. Vậy R3F3M1 là khách mới mở gần đây, mở nhiều lần, nhưng giữ ít sản phẩm.
- **K-means:** để máy tự gom những khách "giống nhau" vào 5 cụm (KM1–KM5), giống gom học sinh có điểm số tương tự vào cùng nhóm. Không cần đặt ngưỡng, nhưng tên cụm phải tự diễn giải.
- **Rule-based:** quy tắc do con người đặt. Ví dụ: mở trong 3 tháng gần nhất và mở ≥ 3 lần → **Champion**; mở gần đây nhưng ít lần → **Active**; lâu không mở (4–6 tháng) → **At-risk**; hơn 6 tháng → **Lost**. Dễ hiểu và dễ giải thích nhất.

### 5. Bốn cách đo RFM
Hai cách đo F (`F_events` đếm số lần mở, `F_event_months` đếm số tháng có mở) × hai cách đo M (`M_n_products` là số sản phẩm, `M_renta` là thu nhập). Cấu hình **chính** là `FM_MNP` = `F_event_months` × `M_n_products`. Tên cấu hình viết tắt: FE/FM cho F, MNP/MREN cho M.

### 6. Hai nguyên tắc tránh
- **Học trước, thi sau:** ngưỡng và tâm cụm chỉ tính từ tháng **08/2015** rồi **giữ nguyên (đóng băng)**, áp lên các tháng sau. Nếu tính từ chính tháng đang chấm, kết quả sẽ đẹp giả tạo (*rò rỉ dữ liệu*).
- **Không nhìn tương lai:** mọi chỉ số của một tháng chỉ dùng dữ liệu tới tháng đó. Mục 3.2 chứng minh bằng cách cắt dữ liệu tương lai đi rồi tính lại, kết quả không đổi.

### 7. Kiểm tra "nhóm có ích không" (mục 7)
Nhìn nhóm của khách hôm nay, có đoán được khách đó **có mở sản phẩm trong 3 tháng sau** không? Các thước đo:
- **Mốc so sánh (baseline):** chỉ chia "đã từng mở" và "chưa từng mở". Nhóm RFM phải thắng mốc này mới có giá trị thêm.
- **Lift@10%:** lấy 10% khách có điểm cao nhất, so tỷ lệ mở của họ với trung bình toàn bộ. Lift = 3 nghĩa là chọn đúng gấp 3 lần so với chọn ngẫu nhiên (1,0 là ngẫu nhiên).
- **PR-AUC/base:** đo chất lượng xếp hạng toàn bộ danh sách; 1,0 là ngẫu nhiên, càng cao càng tốt.
- **Silhouette:** đo các cụm k-means tách nhau rõ đến đâu (0,3–0,4 là trung bình; với RFM thế này là bình thường).
- **Hai loại nhãn:** "mọi event" (gồm cả mở lại sản phẩm đã đóng) và "sản phẩm mới thật sự" (chỉ lần mở đầu tiên của từng sản phẩm).

### 8. Kết quả chính, nói ngắn gọn
- Chọn **`FM_MNP` + quantile** làm nhãn chính; rule-based làm lớp diễn giải; k-means giữ để tham khảo.
- Các nhóm RFM **tốt hơn mốc**, nhưng trong nhóm khách đã có event thì khả năng dự báo chỉ ở mức vừa.
- Khách chưa từng mở thêm sản phẩm (78,7%) rất ít mở trong 3 tháng sau (khoảng 2%), nhưng nhóm trong đó giữ nhiều sản phẩm lại mở khá nhiều (khoảng 11%).
- **Lưu ý quan trọng:** nhóm "tốt nhất" theo RFM (Champion, KM1) chủ yếu hay **mở lại sản phẩm cũ**. Người mở **sản phẩm mới** nhiều nhất là khách vừa mở gần đây nhưng ít lần (nhóm Active, KM4). Muốn nhắm bán chéo thì chọn nhóm này, không mặc định chọn Champion.

### 9. Từ điển nhanh
| Thuật ngữ | Nghĩa |
|---|---|
| Snapshot | ảnh chụp danh mục sản phẩm của khách mỗi tháng |
| Event | khách mở thêm một sản phẩm (cờ 0→1) |
| Proxy | đại lượng thay thế khi không có dữ liệu thật (ở đây thay cho tiền giao dịch) |
| Flicker | khách đóng rồi mở lại cùng sản phẩm liên tục, làm F bị thổi phồng |
| Fit / đóng băng | học ngưỡng một lần ở 08/2015 rồi giữ nguyên |
| Strict | tháng đánh giá mà nhãn của tháng học đã biết từ trước (01 và 02/2016), không bị chồng lấn |
| Baseline | phương án đơn giản nhất để so sánh |
| `No_Acquisition_Event` | khách chưa từng có event, không có R |

### 10. Cách đọc và chạy notebook
- Chạy các ô code **theo thứ tự**. Cần ba file đầu vào: `train_ver2.csv.zip`, `acquisition_events.parquet`, `rfm_proxy_raw.parquet` (hai file sau từ notebook 01).
- Ô bắt đầu bằng 📝 là **chú thích giải thích kết quả** của ô code ngay phía trên.
- Đầu ra chính: `customer_month_features/` (R/F/M từng tháng), `rfm_labels/` (nhãn nhóm từng tháng), `rfm_thresholds.json` (ngưỡng đã học) và các file CSV kết quả ở mục 7.
- Muốn hiểu nhanh: đọc **mục 8** (kết luận) trước, rồi xem các ô 📝 ở mục 6 và 7.

## 0. Cấu hình

In [1]:
import os, glob, zipfile, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

WORK_DIR = os.environ.get('WORK_DIR', '/kaggle/working/')
FEATURE_DIR = Path(WORK_DIR) / 'customer_month_features'
WRITE_MONTHLY_FEATURES = True

# Cohort onboarding hàng loạt (giống notebook 01: cờ theo tháng xuất hiện lần đầu)
ONBOARDING_IDX = 2015 * 12 + 7

# Số liệu notebook 01 ở tháng chốt 05/2016; đổi ở đây nếu chạy trên phiên bản dữ liệu khác
EXPECTED = {'rfm_customers': 931_453, 'events': 561_710}

PRODUCT_MAP = {
    'ind_ahor_fin_ult1': 'prod_saving_account', 'ind_aval_fin_ult1': 'prod_guarantees',
    'ind_cco_fin_ult1': 'prod_current_account', 'ind_cder_fin_ult1': 'prod_derivada_account',
    'ind_cno_fin_ult1': 'prod_payroll_account', 'ind_ctju_fin_ult1': 'prod_junior_account',
    'ind_ctma_fin_ult1': 'prod_mas_particular_account', 'ind_ctop_fin_ult1': 'prod_particular_account',
    'ind_ctpp_fin_ult1': 'prod_particular_plus_account', 'ind_deco_fin_ult1': 'prod_short_term_deposit',
    'ind_deme_fin_ult1': 'prod_medium_term_deposit', 'ind_dela_fin_ult1': 'prod_long_term_deposit',
    'ind_ecue_fin_ult1': 'prod_eaccount', 'ind_fond_fin_ult1': 'prod_funds',
    'ind_hip_fin_ult1': 'prod_mortgage', 'ind_plan_fin_ult1': 'prod_pension_plan',
    'ind_pres_fin_ult1': 'prod_loans', 'ind_reca_fin_ult1': 'prod_taxes',
    'ind_tjcr_fin_ult1': 'prod_credit_card', 'ind_valo_fin_ult1': 'prod_securities',
    'ind_viv_fin_ult1': 'prod_home_account', 'ind_nomina_ult1': 'prod_payroll',
    'ind_nom_pens_ult1': 'prod_pension_payroll', 'ind_recibo_ult1': 'prod_direct_debit',
}
PRODUCT_COLS_RAW = list(PRODUCT_MAP)
assert len(PRODUCT_COLS_RAW) == 24

# Columns written by notebook 01 to acquisition_events.parquet.
EVENT_COLS = ['customer_id', 'record_date', 'product', 'is_onboarding_batch']

RFM_RAW_COLS = ['customer_id', 'M_n_products', 'F_events', 'F_event_months', 'M_renta', 'has_event', 'R',
                'tenure_months', 'renta_missing', 'as_of_month', 'is_onboarding_batch', 'first_idx',
                'n_prod_first', 'zero_at_arrival', 'alta_idx', 'true_tenure_months',
                'opened_before_window', 'arrival_type', 'opened_with_product', 'why_no_event']
def month_idx_of(ts):
    return ts.year * 12 + ts.month

def month_ts(idx):
    return pd.Timestamp(year=(idx - 1) // 12, month=(idx - 1) % 12 + 1, day=28)

def resolve_input(filename):
    local = Path(WORK_DIR) / filename
    if local.exists():
        return str(local)
    matches = glob.glob(f'/kaggle/input/**/{filename}', recursive=True)
    assert len(matches) == 1, f'Cần đúng 1 {filename}; thấy {matches}'
    return matches[0]

---
## 1. Chuẩn hóa đầu vào

- `load_csv_snapshots`: đọc CSV theo chunk, giữ bảng hẹp khách × tháng (số sản phẩm, cờ thiếu, `is_active`, `renta`) + `fecha_alta`. Không giữ 24 cờ sản phẩm.
- `build_window`: cửa sổ quan sát từng khách (`first_idx`, `n_months`, tháng hụt, cờ cohort 07/2015, `arrival_type` từ `fecha_alta`).
- Kiểm tra ba file cùng phiên bản: mỗi event phải có snapshot ở tháng đó **và** tháng liền trước; khách có snapshot ở tháng cuối phải đúng bằng khách trong `rfm_proxy_raw`.

In [2]:
def load_csv_snapshots(zip_path, chunksize=1_000_000):
    """Đọc CSV theo chunk, chỉ giữ bảng hẹp khách × tháng (không giữ 24 cờ sản phẩm).
    Trả về (obs, alta): snapshot tháng và ngày đăng ký theo khách."""
    usecols = ['fecha_dato', 'ncodpers', 'fecha_alta', 'ind_actividad_cliente', 'renta', *PRODUCT_COLS_RAW]
    dtype = {c: 'float32' for c in PRODUCT_COLS_RAW}
    dtype.update({'ncodpers': 'int32', 'renta': 'float32', 'ind_actividad_cliente': 'float32'})
    dtype.update({c: 'str' for c in ['fecha_dato', 'fecha_alta']})
    obs_parts, alta_parts = [], []
    with zipfile.ZipFile(zip_path) as zf, zf.open('train_ver2.csv') as src:
        for ch in pd.read_csv(src, usecols=usecols, dtype=dtype, na_values=['NA', ' NA'], chunksize=chunksize):
            d = pd.to_datetime(ch.fecha_dato.str.strip())
            month = (d.dt.year * 12 + d.dt.month).astype('int32').to_numpy()
            prod = ch[PRODUCT_COLS_RAW]
            n_bad = int((~prod.isin([0, 1]) & prod.notna()).sum().sum())
            assert n_bad == 0, f'{n_bad:,} giá trị sản phẩm ngoài 0/1/NaN'
            obs_parts.append(pd.DataFrame({
                'customer_id': ch.ncodpers.to_numpy(), 'month_idx': month,
                'is_active': ch.ind_actividad_cliente.fillna(-1).astype('int8').to_numpy(),
                'n_products': prod.sum(axis=1, min_count=len(PRODUCT_COLS_RAW)).astype('float32').to_numpy(),
                'portfolio_has_unknown': prod.isna().any(axis=1).to_numpy(),
                'renta': ch.renta.to_numpy()}))
            alta = pd.to_datetime(ch.fecha_alta.str.strip(), errors='coerce')
            alta_parts.append(pd.Series(alta.to_numpy(), index=pd.Index(ch.ncodpers.to_numpy(), name='customer_id')).dropna().groupby(level=0).min())
    obs = pd.concat(obs_parts, ignore_index=True).sort_values(['customer_id', 'month_idx']).reset_index(drop=True)
    assert not obs.duplicated(['customer_id', 'month_idx']).any(), 'Có khóa khách-tháng trùng'
    alta = pd.concat(alta_parts).groupby(level=0).min()
    return obs, alta


def build_window(obs, alta, onboarding_idx=ONBOARDING_IDX):
    """Cửa sổ quan sát của từng khách, tạo trực tiếp từ snapshot (không cần Parquet thứ ba)."""
    g = obs.groupby('customer_id', sort=True).month_idx
    w = pd.DataFrame({'first_idx': g.min(), 'last_idx': g.max(), 'n_months': g.size()}).astype('int32')
    first_month, last_month = int(obs.month_idx.min()), int(obs.month_idx.max())
    w['span_months'] = w.last_idx - w.first_idx + 1
    w['missing_months_in_span'] = w.span_months - w.n_months
    w['is_onboarding_batch'] = w.first_idx.eq(onboarding_idx)
    w['is_censored'] = w.last_idx.eq(last_month)                 # còn thấy ở tháng cuối dataset
    w['months_since_last_seen'] = last_month - w.last_idx
    w['tenure_months'] = w.last_idx - w.first_idx                # tương thích notebook cũ
    a = alta.reindex(w.index)
    w['alta_idx'] = a.dt.year * 12 + a.dt.month                  # NaN nếu thiếu fecha_alta
    w['opened_before_window'] = w.alta_idx.lt(first_month)
    w['arrival_type'] = np.select(
        [w.alta_idx.isna(), w.opened_before_window & w.first_idx.eq(first_month), w.opened_before_window],
        ['unknown', 'legacy_from_start', 'registered_earlier_seen_later'], default='new_in_window')
    w['first_seen'] = pd.to_datetime(pd.DataFrame({'year': (w.first_idx - 1) // 12, 'month': (w.first_idx - 1) % 12 + 1, 'day': 28}))
    w['last_seen'] = pd.to_datetime(pd.DataFrame({'year': (w.last_idx - 1) // 12, 'month': (w.last_idx - 1) % 12 + 1, 'day': 28}))
    return w.reset_index()

In [3]:
RFM_REQUIRED = ['customer_id', 'M_n_products', 'F_events', 'F_event_months', 'M_renta', 'R', 'as_of_month']

def load_notebook01_outputs(events_path, rfm_path):
    """Đọc hai Parquet của notebook 01. Đối soát R/F/M chỉ cần RFM_REQUIRED; bản cũ (còn cột `F`,
    thiếu cột nguồn gốc khách) vẫn dùng được nhưng có cảnh báo."""
    ev = pd.read_parquet(events_path)
    rfm = pd.read_parquet(rfm_path)
    assert list(ev.columns) == EVENT_COLS, f'Schema acquisition_events lệch: {list(ev.columns)}'
    if 'F' in rfm.columns and 'F_events' not in rfm.columns:
        rfm = rfm.rename(columns={'F': 'F_events'})
        print("Cảnh báo: rfm_proxy_raw còn cột `F` (bản cũ) → đọc là `F_events`.")
    missing = set(RFM_REQUIRED) - set(rfm.columns)
    assert not missing, f'rfm_proxy_raw thiếu cột bắt buộc: {sorted(missing)}'
    optional = [c for c in RFM_RAW_COLS if c not in rfm.columns]
    if optional:
        print(f'Cảnh báo: rfm_proxy_raw là bản CŨ, thiếu {len(optional)} cột nguồn gốc khách: {optional}.\n'
              '  Đối soát R/F/M vẫn chạy; để đủ 20 cột hãy chạy hết notebook 01 (tới cell lưu ở mục 9) rồi chạy lại.')
    assert rfm.customer_id.is_unique, 'rfm_proxy_raw: customer_id trùng'
    assert not ev.duplicated(['customer_id', 'record_date', 'product']).any(), 'acquisition_events: dòng trùng'
    assert rfm.as_of_month.nunique() == 1, 'rfm_proxy_raw phải có đúng một as_of_month'
    ev['month_idx'] = (ev.record_date.dt.year * 12 + ev.record_date.dt.month).astype('int32')
    return ev, rfm


def check_inputs_consistent(obs, window, events, rfm_raw):
    """Ba file (CSV, event, rfm) có cùng phiên bản dữ liệu không? Dừng nếu lệch."""
    last_idx = int(obs.month_idx.max())
    as_of = rfm_raw.as_of_month.iloc[0]
    assert month_idx_of(as_of) == last_idx, f'as_of của rfm ({as_of.date()}) khác tháng cuối CSV'
    assert events.customer_id.isin(window.customer_id).all(), 'Có event của khách không có trong CSV'
    assert events.month_idx.gt(int(obs.month_idx.min())).all() and events.month_idx.le(last_idx).all()
    # mỗi event phải có snapshot ở tháng đó VÀ tháng liền trước (đúng quy tắc 0→1 hai tháng liền kề)
    keys = obs.customer_id.to_numpy('int64') * 32768 + obs.month_idx.to_numpy('int64')
    ek = events.customer_id.to_numpy('int64') * 32768 + events.month_idx.to_numpy('int64')
    assert np.isin(ek, keys).all(), 'Event không có snapshot cùng tháng'
    assert np.isin(ek - 1, keys).all(), 'Event không có snapshot tháng liền trước'
    now_ids = set(obs.loc[obs.month_idx.eq(last_idx), 'customer_id'])
    assert now_ids == set(rfm_raw.customer_id), (
        f'Khách có snapshot ở tháng cuối ({len(now_ids):,}) khác khách trong rfm_proxy_raw ({len(rfm_raw):,})')
    print(f'OK: {len(obs):,} snapshot | {len(window):,} khách | {len(events):,} event | '
          f'{len(rfm_raw):,} khách tại as_of={as_of.date()}')


def reconcile_with_notebook01(final_panel, rfm_raw, events, expected=None):
    """Đối soát tháng chốt với rfm_proxy_raw. Trả về bảng khác biệt theo khách/cột; assert khi có lệch."""
    cur = final_panel[final_panel.is_observed]
    m = cur.merge(rfm_raw, on='customer_id', suffixes=('_lc', '_01'), validate='one_to_one')
    assert len(m) == len(cur) == len(rfm_raw), 'Tập khách hai bên không trùng'
    diffs = []
    for c in ['R', 'F_events', 'F_event_months', 'M_n_products', 'M_renta']:
        a, b = m[c + '_lc'], m[c + '_01']
        neq = ~((a == b) | (a.isna() & b.isna()))
        diffs.append(m.loc[neq, ['customer_id']].assign(column=c, notebook02=a[neq], notebook01=b[neq]))
    diff = pd.concat(diffs, ignore_index=True)
    total_f = int(final_panel.F_events.sum())          # gồm cả khách đã vắng ở tháng chốt
    print(f'Khách đối soát: {len(m):,} | tổng F_events (mọi khách đã xuất hiện): {total_f:,} | event trong file: {len(events):,}')
    print('Số khác biệt theo cột:', diff.column.value_counts().to_dict() or 'không có')
    if len(diff):
        display(diff.head(20))
    assert total_f == len(events), 'Tổng F_events không khớp số event'
    if expected:
        assert len(rfm_raw) == expected['rfm_customers'] and len(events) == expected['events'], (
            f"Khác kỳ vọng {expected}: rfm={len(rfm_raw):,}, event={len(events):,}")
    assert diff.empty, 'Feature tháng chốt lệch rfm_proxy_raw (xem bảng trên)'
    print('PASS: R, F_events, F_event_months, M_n_products, M_renta khớp rfm_proxy_raw')
    return diff


def test_no_lookahead(obs, events, alta, cutoff_idx, kept_full):
    """Cắt toàn bộ dữ liệu sau cutoff, tính lại từ đầu; bảng tại cutoff phải giống hệt bản tính trên dữ liệu đầy đủ."""
    obs_c, ev_c = obs[obs.month_idx <= cutoff_idx], events[events.month_idx <= cutoff_idx]
    win_c = build_window(obs_c, alta)
    _, _, final_c = build_monthly_features(obs_c, ev_c, win_c)
    pd.testing.assert_frame_equal(final_c.reset_index(drop=True), kept_full.reset_index(drop=True))
    print(f'PASS: bảng tại {month_ts(cutoff_idx):%Y-%m} giống hệt sau khi cắt dữ liệu tương lai ({len(final_c):,} dòng)')

In [4]:
def build_monthly_features(obs, events, window, output_dir=None, keep_month=None):
    """Bảng feature khách × tháng. Tại tháng m chỉ dùng snapshot/event có month_idx <= m.
    Khách đã xuất hiện lần đầu (first_idx <= m) đều có dòng; tháng vắng: is_observed=False, is_stale=True,
    giá trị danh mục/active/renta là giá trị CŨ từ lần thấy gần nhất.
    Trả về (summary, kept_panel, final_panel)."""
    ids = window.customer_id.to_numpy()
    assert (np.diff(ids) > 0).all(), 'window phải sắp theo customer_id'
    n = len(ids)
    first_idx = window.first_idx.to_numpy('int32')
    lo, hi = int(obs.month_idx.min()), int(obs.month_idx.max())
    assert (events.month_idx <= hi).all() and events.customer_id.isin(window.customer_id).all()

    o_cid, o_act = obs.customer_id.to_numpy(), obs.is_active.to_numpy()
    o_npr = obs.n_products.to_numpy()
    o_unk, o_ren = obs.portfolio_has_unknown.to_numpy(), obs.renta.to_numpy()
    obs_rows = obs.groupby('month_idx').indices
    ev_pos = np.searchsorted(ids, events.customer_id.to_numpy())
    assert (ids[ev_pos] == events.customer_id.to_numpy()).all()
    ev_rows = events.reset_index(drop=True).groupby('month_idx').indices

    last_seen = np.full(n, -1, 'int32'); last_active = np.full(n, -1, 'int8')
    n_prod = np.full(n, np.nan, 'float32'); prod_unknown = np.ones(n, bool)
    renta = np.full(n, np.nan, 'float32'); freq = np.zeros(n, 'int32'); ev_months = np.zeros(n, 'int32')
    last_event = np.full(n, -1, 'int32'); obs_count = np.zeros(n, 'int16')
    if output_dir is not None:
        Path(output_dir).mkdir(parents=True, exist_ok=True)
    summary, kept, panel = [], None, None

    for m in range(lo, hi + 1):
        observed = np.zeros(n, bool)
        rows = obs_rows.get(m)
        if rows is not None:
            pos = np.searchsorted(ids, o_cid[rows])
            assert (ids[pos] == o_cid[rows]).all()
            observed[pos] = True; obs_count[pos] += 1; last_seen[pos] = m
            last_active[pos] = o_act[rows]
            n_prod[pos] = o_npr[rows]; prod_unknown[pos] = o_unk[rows]
            r = o_ren[rows]; ok = ~np.isnan(r); renta[pos[ok]] = r[ok]        # M_renta = giá trị hợp lệ gần nhất
        erows = ev_rows.get(m)
        if erows is not None:
            epos = ev_pos[erows]
            assert observed[epos].all(), 'Event phải có snapshot cùng tháng'
            cnt = np.bincount(epos, minlength=n).astype('int32')
            freq += cnt; ev_months += (cnt > 0); last_event[epos] = m

        el = np.flatnonzero(first_idx <= m)
        le = last_event[el]
        panel = pd.DataFrame({
            'customer_id': ids[el],
            'as_of_month': np.full(len(el), np.datetime64(month_ts(m), 'us'), dtype='datetime64[us]'),
            'is_observed': observed[el],
            'is_stale': ~observed[el],
            'months_since_last_seen': (m - last_seen[el]).astype('int16'),
            'is_active': last_active[el],
            'M_n_products': n_prod[el], 'portfolio_has_unknown': prod_unknown[el],
            'M_renta': renta[el], 'renta_missing': np.isnan(renta[el]),
            'R': np.where(le >= 0, m - le, np.nan),
            'F_events': freq[el], 'F_event_months': ev_months[el], 'has_event': freq[el] > 0,
            'observed_months': obs_count[el], 'tenure_months': (m - first_idx[el]).astype('int32'),
        })
        summary.append({'month_idx': m, 'n_eligible': len(el), 'n_observed': int(observed.sum()),
                        'n_stale': int((~observed[el]).sum()), 'n_has_event': int((freq[el] > 0).sum())})
        if output_dir is not None:
            panel.to_parquet(Path(output_dir) / f'customer_features_{month_ts(m):%Y-%m}.parquet', index=False)
        if keep_month == m:
            kept = panel
    return pd.DataFrame(summary).set_index('month_idx'), kept, panel

---
## 2. Đọc dữ liệu thật và kiểm tra phiên bản

In [5]:
zip_path = resolve_input('train_ver2.csv.zip')
obs, alta = load_csv_snapshots(zip_path)
window = build_window(obs, alta)
events, rfm_raw = load_notebook01_outputs(resolve_input('acquisition_events.parquet'),
                                          resolve_input('rfm_proxy_raw.parquet'))
FIRST_IDX, LAST_IDX = int(obs.month_idx.min()), int(obs.month_idx.max())
print(f'{month_ts(FIRST_IDX):%Y-%m} → {month_ts(LAST_IDX):%Y-%m} | {len(window):,} khách | '
      f'còn thấy ở tháng cuối: {int(window.is_censored.sum()):,} | có tháng hụt: {int((window.missing_months_in_span > 0).sum()):,} | '
      f'thiếu fecha_alta: {int(window.alta_idx.isna().sum()):,}')
check_inputs_consistent(obs, window, events, rfm_raw)

# window (tạo từ CSV) phải khớp các cột cùng tên trong rfm_proxy_raw của notebook 01
m = window.merge(rfm_raw, on='customer_id', suffixes=('_w', '_01'), validate='one_to_one')
cmp_cols = [c for c in ['first_idx', 'is_onboarding_batch', 'arrival_type', 'opened_before_window'] if c in rfm_raw.columns]
for c in cmp_cols:
    n_diff = int((m[c + '_w'] != m[c + '_01']).sum())
    print(f'{c:<22} lệch {n_diff:,}')
    assert n_diff == 0, f'window (CSV) lệch rfm_proxy_raw ở cột {c}'
if 'alta_idx' in rfm_raw.columns:
    n_alta = int((m.alta_idx_w.fillna(-1) != m.alta_idx_01.fillna(-1)).sum())
    assert n_alta == 0, f'alta_idx lệch {n_alta:,}'
if len(cmp_cols) < 4 or 'alta_idx' not in rfm_raw.columns:
    print('Bỏ qua so sánh các cột nguồn gốc khách vì rfm_proxy_raw là bản cũ.')
del m
display(window.arrival_type.value_counts().to_frame('n_khach'))

Cảnh báo: rfm_proxy_raw còn cột `F` (bản cũ) → đọc là `F_events`.
Cảnh báo: rfm_proxy_raw là bản CŨ, thiếu 9 cột nguồn gốc khách: ['first_idx', 'n_prod_first', 'zero_at_arrival', 'alta_idx', 'true_tenure_months', 'opened_before_window', 'arrival_type', 'opened_with_product', 'why_no_event'].
  Đối soát R/F/M vẫn chạy; để đủ 20 cột hãy chạy hết notebook 01 (tới cell lưu ở mục 9) rồi chạy lại.
2015-01 → 2016-05 | 956,645 khách | còn thấy ở tháng cuối: 931,453 | có tháng hụt: 8,018 | thiếu fecha_alta: 7,031
OK: 13,647,309 snapshot | 956,645 khách | 561,710 event | 931,453 khách tại as_of=2016-05-28
is_onboarding_batch    lệch 0
Bỏ qua so sánh các cột nguồn gốc khách vì rfm_proxy_raw là bản cũ.


,n_khach
arrival_type,
legacy_from_start,616597
registered_earlier_seen_later,177954
new_in_window,155063
unknown,7031


> ### 📝 Chú thích kết quả (mục 2)
> - **13,65 triệu snapshot / 956.645 khách / 561.710 event**, giai đoạn 2015-01 → 2016-05. Có 931.453 khách còn xuất hiện ở tháng cuối (25.192 khách đã vắng).
> - Hai cảnh báo là do `rfm_proxy_raw` là bản cũ (còn cột `F`, thiếu 9 cột nguồn gốc). R/F/M vẫn đối soát được nên không ảnh hưởng kết quả, nhưng nên chạy lại notebook 01 cho đủ 20 cột.
> - **`arrival_type`:** 64% khách là `legacy_from_start` (đã có từ tháng đầu), 16% là `new_in_window`, 18,6% là `registered_earlier_seen_later` (cohort xuất hiện hàng loạt 07/2015). Nhóm cuối này dễ làm lệch tenure và nhiễu R/F nếu không tách riêng.
> - 7.031 khách thiếu `fecha_alta` → `unknown`.

---
## 3. Feature khách × tháng

Mỗi tháng chỉ dùng snapshot/event có `month_idx <= tháng đó`; trạng thái mang sang tháng sau bằng mảng theo khách. Ghi từng Parquet theo tháng để không giữ cả bảng mở rộng trong RAM.

In [6]:
CUTOFF_IDX = FIRST_IDX + 11          # 12/2015: tháng dùng cho test không nhìn tương lai
feature_summary, kept_cutoff, final_panel = build_monthly_features(
    obs, events, window, output_dir=FEATURE_DIR if WRITE_MONTHLY_FEATURES else None, keep_month=CUTOFF_IDX)
display(feature_summary)
print('Output tháng:', FEATURE_DIR if WRITE_MONTHLY_FEATURES else 'tạm tắt để thử nghiệm')

,n_eligible,n_observed,n_stale,n_has_event
month_idx,,,,
24181,625457,625457,0,0
24182,629925,627394,2531,23782
24183,633901,629209,4692,46298
24184,637324,630367,6957,65298
24185,640825,631957,8868,77607
24186,644042,632110,11932,93542
24187,837157,829817,7340,105077
24188,851956,843201,8755,117197
24189,875554,865440,10114,128837


Output tháng: /kaggle/working/customer_month_features


> ### 📝 Chú thích bảng tóm tắt khách × tháng (mục 3)
> - **`n_eligible`:** số khách đã xuất hiện ít nhất một lần tới tháng đó, tăng dần từ 625.457 lên 956.645.
> - **Bước nhảy 07/2015 (644.042 → 837.157):** cohort onboarding hàng loạt, thêm khoảng 193 nghìn khách trong một tháng. Đây là lý do `ONBOARDING_IDX` được tách riêng.
> - **`n_stale`** tăng đều (0 → 25.192): khách vắng ngày càng nhiều, giá trị của họ là giá trị cũ, không nên coi là sở hữu hiện tại.
> - **`n_has_event`** tăng 0 → 201.796, nhưng đây là số tích lũy (đã từng có event), không phải số khách đang hoạt động.

### 3.1. Đối soát với notebook 01 (tháng 05/2016)

Trên khách có snapshot ở tháng cuối: `R`, `F_events`, `F_event_months`, `M_n_products`, `M_renta` phải khớp `rfm_proxy_raw.parquet`; tổng `F_events` (kể cả khách đã vắng) phải bằng số event; kỳ vọng 931.453 khách và 561.710 event. Khác biệt được liệt kê theo khách/cột **trước** khi đi tiếp.

In [7]:
diff_vs_01 = reconcile_with_notebook01(final_panel, rfm_raw, events, EXPECTED)

Khách đối soát: 931,453 | tổng F_events (mọi khách đã xuất hiện): 561,710 | event trong file: 561,710
Số khác biệt theo cột: không có
PASS: R, F_events, F_event_months, M_n_products, M_renta khớp rfm_proxy_raw


> ### 📝 Chú thích đối soát với notebook 01
> - 931.453 khách khớp 100% với `rfm_proxy_raw` ở cả 5 cột (R, F_events, F_event_months, M_n_products, M_renta) và tổng F_events = 561.710 = số event trong file.
> - Nghĩa là pipeline feature theo tháng **tái lập đúng** bảng RFM gốc, an toàn để dùng làm đầu vào cho Đợt 2.

### 3.2. Không nhìn tương lai

Cắt toàn bộ snapshot/event sau 12/2015, dựng lại window và feature từ đầu; bảng tháng 12/2015 phải giống hệt bản tính trên dữ liệu đầy đủ.

In [8]:
test_no_lookahead(obs, events, alta, CUTOFF_IDX, kept_cutoff)

PASS: bảng tại 2015-12 giống hệt sau khi cắt dữ liệu tương lai (927,578 dòng)


> ### 📝 Chú thích kiểm tra không nhìn tương lai
> - Cắt dữ liệu sau 12/2015, dựng lại từ đầu thì bảng tại 12/2015 (927.578 dòng) giống hệt bản tính trên dữ liệu đầy đủ.
> - Kết luận: **không rò rỉ thông tin tương lai** vào feature, điều kiện cần để kiểm chứng dự báo ở mục 7.

### 3.3. Xem nhanh output

In [9]:
print(f'Bảng feature tháng cuối ({final_panel.as_of_month.iloc[0]:%Y-%m}): {len(final_panel):,} dòng × {final_panel.shape[1]} cột')
display(pd.DataFrame({'dtype': final_panel.dtypes.astype(str), 'n_missing': final_panel.isna().sum()}))
display(final_panel.sample(8, random_state=0))

Bảng feature tháng cuối (2016-05): 956,645 dòng × 16 cột


,dtype,n_missing
customer_id,int32,0
as_of_month,datetime64[us],0
is_observed,bool,0
is_stale,bool,0
months_since_last_seen,int16,0
is_active,int8,0
M_n_products,float32,5915
portfolio_has_unknown,bool,0
M_renta,float32,247234
renta_missing,bool,0


,customer_id,as_of_month,is_observed,is_stale,months_since_last_seen,is_active,M_n_products,portfolio_has_unknown,M_renta,renta_missing,R,F_events,F_event_months,has_event,observed_months,tenure_months
127562,270152,2016-05-28,True,False,0,1,3.0,False,155198.609375,False,0.0,1,1,True,17,16
337141,704477,2016-05-28,True,False,0,0,1.0,False,80587.500000,False,NaN,0,0,False,17,16
133558,281820,2016-05-28,True,False,0,1,1.0,False,301757.718750,False,NaN,0,0,False,17,16
805904,1376163,2016-05-28,True,False,0,1,1.0,False,160858.203125,False,6.0,1,1,True,17,16
239686,489479,2016-05-28,True,False,0,0,1.0,False,74363.187500,False,NaN,0,0,False,17,16
740056,1300802,2016-05-28,True,False,0,0,1.0,False,83406.031250,False,NaN,0,0,False,17,16
735336,1294624,2016-05-28,True,False,0,0,1.0,False,162324.296875,False,NaN,0,0,False,17,16
288024,597839,2016-05-28,True,False,0,0,0.0,False,225990.750000,False,NaN,0,0,False,17,16


> ### 📝 Chú thích bảng feature tháng cuối (2016-05)
> - **Thiếu dữ liệu:** `M_renta` thiếu 247.234 dòng (25,8%), `M_n_products` thiếu 5.915 (0,6%, do thiếu cờ sản phẩm). Đây là nguyên nhân cấu hình dùng `M_renta` bị mất độ phủ.
> - **`R` thiếu 754.849 dòng (78,9%):** là khách chưa từng có event 0→1, R = NaN có chủ đích chứ không phải lỗi. Họ không bị coi là churn.
> - Mẫu 8 dòng: `F_events = 0` đi cùng `R = NaN`, còn khách có event (vd. R = 0 hoặc 6) luôn có `F_event_months ≥ 1`.

---
## 4. Đợt 2: bốn cấu hình RFM × ba phương pháp

**Mốc thời gian (theo kế hoạch):** học ngưỡng/tâm cụm trên bảng feature **08/2015** rồi đóng băng; áp lên 09/2015 (chọn cấu hình), 01–04/2016 (kiểm thử cuối) và 05/2016 (tháng chốt xuất cuối kỳ). Không tính lại ngưỡng ở tháng sau.

**Cấu hình:** `FE_MNP` = `F_events × M_n_products`, `FE_MREN` = `F_events × M_renta`, `FM_MNP` = `F_event_months × M_n_products` (**chính**), `FM_MREN` = `F_event_months × M_renta`. Hai cấu hình phụ `FE_MRENfill` / `FM_MRENfill` dùng income được điền bằng median của tập fit, lưu ở cột riêng `M_renta_filled` (không ghi đè `M_renta`).

**Ba phương pháp:**
- **Quantile:** ngưỡng theo *giá trị* (tertile lấy giá trị thực, `method='lower'`), khách cùng giá trị luôn cùng tier. Nhiều giá trị trùng thì ra ít tier hơn, không ép cân bằng. R, F và M cùng học trên khách có event (đúng quần thể được chấm).
- **K-means:** đặc trưng `R`, `log1p(F)`, `M` (income: `log1p` sau khi cắt p1–p99 của tập fit); chuẩn hóa bằng mean/std của tập fit; **k cố định = 5** cho mọi cấu hình (`K_FIXED`; silhouette k = 3–8 vẫn được ghi để báo cáo vì chênh lệch giữa các k quá nhỏ để chọn theo từng cấu hình); `KM1` = điểm tổng −R+F+M cao nhất. Thứ hạng chỉ có nghĩa ở hai đầu, các nhóm giữa không so hơn kém được.
- **Rule-based:** `Champion / Active / At-risk / Lost / No_Acquisition_Event` từ R/F; M là trục riêng (`M_low/M_mid/M_high/Unknown`); đánh giá M bằng cột `rb_segment_x_m`.

**Quy ước chung:** chỉ chấm khách có snapshot ở tháng chốt. `is_scored` = có event **và** có M. Không có event → nhãn `No_Acquisition_Event` (R=NaN, không coi là ít giá trị hay churn). Thiếu M → M tier = `<NA>`, band `Unknown`, k-means không chấm (`Unscored_M_missing`).

**Ngưỡng rule-based:** `RULE_THRESHOLDS` (R ≤ 3 tháng là gần đây, R > 6 là Lost, F ≥ 3 là Champion) và các mốc M (≤1 / 2–3 / ≥4 sản phẩm) được chọn sau EDA sơ bộ phân phối R, F, M. Hai mốc R = 3 và F = 3 trùng với mốc cắt của quantile. Việc tinh chỉnh ngưỡng kỹ hơn được để ở phần mở rộng (mục 9).


In [10]:
import json, copy
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

RFM_METHOD_VERSION = 'rfm-d2.1'
RFM_LABEL_DIR = Path(WORK_DIR) / 'rfm_labels'
WRITE_RFM_LABELS = True

# Mốc thời gian theo kế hoạch: học ngưỡng đến 08/2015; chọn cấu hình ở 09/2015; kiểm thử cuối 01/2016 → 04/2016
FIT_IDX = 2015 * 12 + 8
SELECT_IDX = 2015 * 12 + 9
TEST_IDXS = [2016 * 12 + m for m in (1, 2, 3, 4)]

# Bốn cấu hình chính + hai cấu hình phụ dùng income được điền (cột riêng M_renta_filled)
RFM_CONFIGS = {
    'FE_MNP':  {'F': 'F_events',       'M': 'M_n_products'},
    'FE_MREN': {'F': 'F_events',       'M': 'M_renta'},
    'FM_MNP':  {'F': 'F_event_months', 'M': 'M_n_products'},   # cấu hình chính (kết luận notebook 01)
    'FM_MREN': {'F': 'F_event_months', 'M': 'M_renta'},
}
SENSITIVITY_CONFIGS = {
    'FE_MRENfill': {'F': 'F_events',       'M': 'M_renta_filled'},
    'FM_MRENfill': {'F': 'F_event_months', 'M': 'M_renta_filled'},
}
ALL_CONFIGS = {**RFM_CONFIGS, **SENSITIVITY_CONFIGS}
PRIMARY_CONFIG = 'FM_MNP'

K_RANGE = range(3, 9)                # k-means: k từ 3–8
KM_SEED = 42
K_FIXED = 5                          # k cố định cho mọi cấu hình (None = chọn theo silhouette). Silhouette k=3–8 vẫn được ghi để báo cáo.
KM_SCAN_ROWS, KM_SIL_ROWS = 100_000, 20_000

# Ngưỡng rule-based: chọn sau EDA sơ bộ phân phối R, F, M (tinh chỉnh thêm: xem mục 9).
RULE_THRESHOLDS = {'r_recent': 3,      # R ≤ 3 tháng: còn gần đây (Active / Champion)
                   'r_atrisk': 6,      # 3 < R ≤ 6: At-risk; R > 6: Lost
                   'f_champion': 3}    # Champion: gần đây và F ≥ 3
COUNT_BANDS = (1, 3)                   # M_n_products: 0–1 / 2–3 / ≥4

NO_EVENT = 'No_Acquisition_Event'
LABEL_COLS = ['customer_id', 'as_of_month', 'config', 'is_scored', 'q_R', 'q_F', 'q_M', 'q_segment',
              'km_rank', 'km_label', 'rb_segment', 'rb_m_band', 'rb_segment_x_m', 'method_version']


def is_income(m_col):
    return 'renta' in m_col


In [11]:
# ---------------------------------------------------------------- khung chấm điểm
def scoring_frame(panel, fill_value=None):
    """Chỉ khách có snapshot ở tháng chốt. M_renta_filled là cột riêng; M_renta gốc giữ nguyên (NaN không bị ghi đè)."""
    df = panel[panel.is_observed].reset_index(drop=True)
    if fill_value is not None:
        df = df.assign(M_renta_filled=df.M_renta.fillna(fill_value), renta_was_filled=df.M_renta.isna())
    return df


# ---------------------------------------------------------------- quantile theo giá trị
def value_cuts(x, n_tiers=3):
    """Ngưỡng theo GIÁ TRỊ (không rank): khách cùng giá trị luôn cùng tier. Nhiều giá trị trùng → ít tier hơn, không ép cân bằng."""
    x = np.asarray(x, dtype='float64'); x = x[~np.isnan(x)]
    if x.size == 0:
        return []
    q = np.quantile(x, np.linspace(0, 1, n_tiers + 1)[1:-1], method='lower')
    return sorted({float(v) for v in q})


def tier_of(x, cuts, invert=False):
    """Tier 1..len(cuts)+1 (cao = tốt). x NaN → <NA>. invert=True dùng cho R (nhỏ = tốt)."""
    x = np.asarray(x, dtype='float64')
    idx = np.searchsorted(np.asarray(cuts, dtype='float64'), x, side='left')     # số ngưỡng < x
    t = (len(cuts) + 1 - idx) if invert else (idx + 1)
    return pd.array(np.where(np.isnan(x), np.nan, t), dtype='Int8')


def fit_quantile(fit, f_col, m_col):
    ev = fit[fit.has_event]
    # R, F, M cùng học trên khách có event: đúng quần thể sẽ được chấm (tránh M tertile bị kéo bởi khách không event)
    return {'R_cuts': value_cuts(ev.R), 'F_cuts': value_cuts(ev[f_col]), 'M_cuts': value_cuts(ev[m_col])}


def apply_quantile(df, p, f_col, m_col):
    has = df.has_event.to_numpy()
    r = tier_of(np.where(has, df.R, np.nan), p['R_cuts'], invert=True)
    f = tier_of(np.where(has, df[f_col], np.nan), p['F_cuts'])
    m = tier_of(df[m_col], p['M_cuts'])
    txt = lambda a: pd.Series(a).astype('string').fillna('?')
    seg = np.where(has, 'R' + txt(r) + 'F' + txt(f) + 'M' + txt(m), NO_EVENT)
    return pd.DataFrame({'q_R': r, 'q_F': f, 'q_M': m, 'q_segment': pd.Categorical(seg)})


# ---------------------------------------------------------------- k-means
def _km_raw(df, f_col, m_col, p):
    M = df[m_col].to_numpy('float64')
    if is_income(m_col):
        M = np.log1p(np.clip(M, *p['M_clip']))                                   # log1p + winsorize theo tập fit
    return np.column_stack([df.R.to_numpy('float64'), np.log1p(df[f_col].to_numpy('float64')), M])


def _nearest(Z, C, chunk=400_000):
    out = np.empty(len(Z), 'int16')
    for i in range(0, len(Z), chunk):
        z = Z[i:i + chunk]
        out[i:i + chunk] = ((z[:, None, :] - C[None, :, :]) ** 2).sum(-1).argmin(1)
    return out


def fit_kmeans(fit, f_col, m_col, seed=KM_SEED):
    pop = fit[fit.has_event & fit[m_col].notna()]
    assert len(pop) > max(K_RANGE) * 10, f'Quá ít khách để chạy k-means ({len(pop)})'
    p = {'n_fit': int(len(pop))}
    if is_income(m_col):
        p['M_clip'] = [float(np.percentile(pop[m_col], 1)), float(np.percentile(pop[m_col], 99))]
    X = _km_raw(pop, f_col, m_col, p)
    sd = X.std(0)
    p['mean'], p['std'] = X.mean(0), np.where(sd > 0, sd, 1.0)                    # chuẩn hóa chỉ trên tập fit
    Z = (X - p['mean']) / p['std']
    rng = np.random.default_rng(seed)
    scan = Z if len(Z) <= KM_SCAN_ROWS else Z[rng.choice(len(Z), KM_SCAN_ROWS, replace=False)]
    scores = {}
    for k in K_RANGE:
        km = KMeans(k, n_init=3, random_state=seed).fit(scan)
        scores[int(k)] = {'silhouette': float(silhouette_score(scan, km.labels_, sample_size=min(KM_SIL_ROWS, len(scan)), random_state=seed)),
                          'inertia': float(km.inertia_)}
    k_best = max(scores, key=lambda j: scores[j]['silhouette'])
    k = K_FIXED if K_FIXED else k_best
    assert k in scores, f'K_FIXED={k} ngoài K_RANGE'
    km = KMeans(k, n_init=5, random_state=seed).fit(Z)
    C = km.cluster_centers_
    order = np.argsort(-(-C[:, 0] + C[:, 1] + C[:, 2]))                           # rank 1 = gần đây, F cao, M cao
    rank = np.empty(k, int); rank[order] = np.arange(1, k + 1)
    orig = C * p['std'] + p['mean']
    p.update(k=int(k), k_best_silhouette=int(k_best), k_rule='fixed' if K_FIXED else 'silhouette', scan=scores, centers=C, rank=rank,
             centroids_orig=pd.DataFrame({'rank': rank, 'R': orig[:, 0], 'F': np.expm1(orig[:, 1]),
                                          'M': np.expm1(orig[:, 2]) if is_income(m_col) else orig[:, 2],
                                          'n_fit': np.bincount(km.labels_, minlength=k)}).sort_values('rank').to_dict('list'))
    return p


def apply_kmeans(df, p, f_col, m_col):
    has, m_ok = df.has_event.to_numpy(), df[m_col].notna().to_numpy()
    ok = has & m_ok
    rank = np.full(len(df), np.nan)
    if ok.any():
        Z = (_km_raw(df[ok], f_col, m_col, p) - p['mean']) / p['std']
        rank[ok] = p['rank'][_nearest(Z, p['centers'])]
    label = np.where(ok, 'KM' + pd.Series(rank).astype('Int64').astype('string').fillna(''),
                     np.where(~has, NO_EVENT, 'Unscored_M_missing'))
    return pd.DataFrame({'km_rank': pd.array(rank, dtype='Int8'), 'km_label': pd.Categorical(label)})


# ---------------------------------------------------------------- rule-based
def fit_rule(fit, m_col):
    p = dict(RULE_THRESHOLDS)
    if is_income(m_col):
        p['M_cuts'] = value_cuts(fit.loc[fit.has_event, m_col])   # 3 band theo ngưỡng thu nhập học từ khách có event ở tập fit
    else:
        p['M_cuts'] = [float(COUNT_BANDS[0]), float(COUNT_BANDS[1])]
    return p


def _m_band(M, cuts):
    names = {2: ['M_low', 'M_mid', 'M_high'], 1: ['M_low', 'M_high'], 0: ['M_all']}[len(cuts)]
    M = np.asarray(M, 'float64')
    idx = np.searchsorted(np.asarray(cuts, 'float64'), M, side='left')
    return np.where(np.isnan(M), 'Unknown', np.asarray(names, dtype=object)[np.minimum(idx, len(names) - 1)])


def apply_rule(df, p, f_col, m_col):
    has, R, F = df.has_event.to_numpy(), df.R.to_numpy('float64'), df[f_col].to_numpy('float64')
    seg = np.select([~has, R > p['r_atrisk'], (R <= p['r_recent']) & (F >= p['f_champion']), R <= p['r_recent']],
                    [NO_EVENT, 'Lost', 'Champion', 'Active'], default='At-risk')
    band = _m_band(df[m_col], p['M_cuts'])
    return pd.DataFrame({'rb_segment': pd.Categorical(seg), 'rb_m_band': pd.Categorical(band),
                         'rb_segment_x_m': pd.Categorical(pd.Series(seg).astype(str) + '|' + pd.Series(band).astype(str))})


# ---------------------------------------------------------------- fit / apply toàn bộ cấu hình
def fit_rfm_models(fit_panel, configs=ALL_CONFIGS, fit_idx=None):
    """Học ngưỡng/tâm cụm CHỈ từ bảng feature của tháng fit (dữ liệu ≤ tháng đó)."""
    obs = fit_panel[fit_panel.is_observed]
    fill = float(obs.M_renta.median())
    fit = scoring_frame(fit_panel, fill)
    params = {'meta': {'method_version': RFM_METHOD_VERSION, 'fit_month': str(fit_panel.as_of_month.iloc[0].date()),
                       'fit_idx': fit_idx, 'n_fit_customers': int(len(fit)), 'renta_fill_value': fill},
              'configs': {}}
    for name, c in configs.items():
        params['configs'][name] = {**c, 'quantile': fit_quantile(fit, c['F'], c['M']),
                                   'kmeans': fit_kmeans(fit, c['F'], c['M']), 'rule': fit_rule(fit, c['M'])}
    return params


def apply_rfm_models(panel, params, names=None):
    """Áp mô hình đã đóng băng lên một tháng. Không sửa params."""
    df = scoring_frame(panel, params['meta']['renta_fill_value'])
    out = {}
    for name in (names or params['configs']):
        c = params['configs'][name]
        f_col, m_col = c['F'], c['M']
        lab = pd.concat([apply_quantile(df, c['quantile'], f_col, m_col), apply_kmeans(df, c['kmeans'], f_col, m_col),
                         apply_rule(df, c['rule'], f_col, m_col)], axis=1)
        lab.insert(0, 'is_scored', (df.has_event & df[m_col].notna()).to_numpy())
        lab.insert(0, 'config', pd.Categorical([name] * len(df)))
        lab.insert(0, 'as_of_month', df.as_of_month.to_numpy())
        lab.insert(0, 'customer_id', df.customer_id.to_numpy())
        lab['method_version'] = RFM_METHOD_VERSION
        out[name] = lab[LABEL_COLS]
    return out, df


def params_to_json(params, path):
    def conv(o):
        if isinstance(o, dict): return {str(k): conv(v) for k, v in o.items()}
        if isinstance(o, (list, tuple)): return [conv(v) for v in o]
        if isinstance(o, np.ndarray): return o.tolist()
        if isinstance(o, (np.integer,)): return int(o)
        if isinstance(o, (np.floating,)): return float(o)
        return o
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(conv(params), ensure_ascii=False, indent=1))


# ---------------------------------------------------------------- chẩn đoán
LABEL_BY_METHOD = {'quantile': 'q_segment', 'kmeans': 'km_label', 'rule': 'rb_segment_x_m'}


def coverage_row(lab, df, month_idx, config):
    n = len(lab)
    return {'month_idx': month_idx, 'config': config, 'n_observed': n,
            'pct_has_event': float(df.has_event.mean()), 'pct_M_known': float(df[ALL_CONFIGS[config]['M']].notna().mean()),
            'pct_scored': float(lab.is_scored.mean())}


def stability_rows(prev, cur, month_idx, config):
    """Cùng khách được chấm ở hai tháng liên tiếp: tỷ lệ giữ nguyên nhãn."""
    a, b = prev.set_index('customer_id'), cur.set_index('customer_id')
    ids = a.index[a.is_scored.to_numpy()].intersection(b.index[b.is_scored.to_numpy()])
    a, b = a.loc[ids], b.loc[ids]
    return [{'month_idx': month_idx, 'config': config, 'method': m, 'n_common': int(len(ids)),
             'pct_same_label': float((a[col].astype(str) == b[col].astype(str)).mean())}
            for m, col in LABEL_BY_METHOD.items()]


def switch_report(lab_a, lab_b, df, col='q_segment', top=10):
    """Khách đổi nhóm khi thay cấu hình a → b, trên CÙNG tập khách được chấm ở cả hai."""
    m = lab_a[['customer_id', col, 'is_scored']].merge(lab_b[['customer_id', col, 'is_scored']], on='customer_id',
                                                       suffixes=('_a', '_b'), validate='one_to_one')
    m = m[m.is_scored_a & m.is_scored_b]
    m['changed'] = m[col + '_a'].astype(str) != m[col + '_b'].astype(str)
    prof = m.merge(df[['customer_id', 'R', 'F_events', 'F_event_months', 'M_n_products', 'M_renta']], on='customer_id')
    prof['F_gap'] = prof.F_events - prof.F_event_months
    profile = prof.groupby('changed')[['R', 'F_events', 'F_event_months', 'F_gap', 'M_n_products', 'M_renta']].mean().round(2)
    flows = (m[m.changed].groupby([col + '_a', col + '_b'], observed=True).size().sort_values(ascending=False)
             .head(top).rename('n_khach').reset_index())
    return {'n_common': int(len(m)), 'n_changed': int(m.changed.sum()), 'pct_changed': float(m.changed.mean()),
            'profile': profile, 'top_flows': flows}


# ---------------------------------------------------------------- mô tả nhóm không event, tương quan M, kiểm flicker
def no_event_profile(df, window):
    """Mô tả khách có snapshot ở tháng chốt theo có/không event: cấu trúc cohort, số sản phẩm, active, thiếu income."""
    d = df.merge(window[['customer_id', 'arrival_type']], on='customer_id', how='left')
    d['nhom'] = np.where(d.has_event, 'Có event', NO_EVENT)
    d['M_band'] = pd.cut(d.M_n_products, [-1, 1, 3, np.inf], labels=['0-1 sp', '2-3 sp', '>=4 sp']).astype(object).fillna('Unknown')
    out = {'quy_mo': d.groupby('nhom').size().to_frame('n_khach').assign(pct=lambda t: (t.n_khach / t.n_khach.sum()).round(4))}
    for col in ['arrival_type', 'M_band']:
        out[col] = pd.crosstab(d.nhom, d[col], normalize='index').round(3)
    out['chi_so'] = d.groupby('nhom').agg(pct_active=('is_active', lambda s: (s == 1).mean()), pct_thieu_renta=('M_renta', lambda s: s.isna().mean()),
                                          tenure_tb=('tenure_months', 'mean'), n_sp_tb=('M_n_products', 'mean')).round(3)
    return out


def m_correlation(df):
    """Tương quan hạng giữa hai trục M (số sản phẩm vs thu nhập): thấp = hai trục đo hai thứ khác nhau."""
    rows = []
    for name, sub in [('mọi khách có snapshot', df), ('khách có event', df[df.has_event])]:
        s = sub[['M_n_products', 'M_renta']].dropna()
        rows.append({'tập': name, 'n': len(s), 'spearman': round(float(s.M_n_products.corr(s.M_renta, method='spearman')), 3)})
    return pd.DataFrame(rows).set_index('tập')


def flicker_check(events, df, f_min=3, m_hi=1):
    """Khách F_event_months >= f_min nhưng đang giữ <= m_hi sản phẩm: bao nhiêu event là MỞ LẠI cùng một sản phẩm?
    repeat_share = 1 - (số sản phẩm khác nhau / số event). So với nhóm F cao nhưng M >= m_hi+1."""
    hi = df[df.F_event_months >= f_min]
    grp = {'F cao, M thấp': hi[hi.M_n_products <= m_hi], 'F cao, M cao': hi[hi.M_n_products > m_hi]}
    ev = events[events.month_idx <= int(df.as_of_month.iloc[0].year * 12 + df.as_of_month.iloc[0].month)]
    per = ev.groupby('customer_id').agg(n_event=('product', 'size'), n_distinct=('product', 'nunique'))
    rows = []
    for name, g in grp.items():
        p = per.reindex(g.customer_id).dropna()
        rows.append({'nhom': name, 'n_khach': len(g), 'event_tb': round(float(p.n_event.mean()), 2) if len(p) else np.nan,
                     'sp_khac_nhau_tb': round(float(p.n_distinct.mean()), 2) if len(p) else np.nan,
                     'pct_khach_co_mo_lai': round(float((p.n_event > p.n_distinct).mean()), 3) if len(p) else np.nan,
                     'repeat_share': round(float(1 - p.n_distinct.sum() / p.n_event.sum()), 3) if len(p) else np.nan})
    return pd.DataFrame(rows).set_index('nhom')


---
## 5. Chạy trên dữ liệu thật

Đọc lại từng `customer_features_YYYY-MM.parquet` từ mục 4 (cần `WRITE_MONTHLY_FEATURES=True`). Mỗi tháng chỉ giữ nhãn tháng trước (đo ổn định) và nhãn tháng fit / tháng cuối (so sánh đổi nhóm).

In [12]:
def load_month_panel(idx):
    p = FEATURE_DIR / f'customer_features_{month_ts(idx):%Y-%m}.parquet'
    assert p.exists(), f'Thiếu {p}. Chạy mục 4 với WRITE_MONTHLY_FEATURES=True trước.'
    return pd.read_parquet(p)


def run_rfm_experiments(label_idxs, fit_idx, last_idx, write=True):
    """Học ngưỡng ở tháng fit, đóng băng, áp lên từng tháng chốt. Chỉ giữ trong RAM nhãn của tháng trước (để đo ổn định)
    và nhãn của tháng fit / tháng cuối (để so sánh đổi nhóm). Kích thước nhóm được ghi cho mọi tháng (đo độ trôi)."""
    params = fit_rfm_models(load_month_panel(fit_idx), fit_idx=fit_idx)
    cov, sizes, stab, keep, prev = [], [], [], {}, None
    for idx in label_idxs:
        labs, df = apply_rfm_models(load_month_panel(idx), params)
        for name, lab in labs.items():
            cov.append(coverage_row(lab, df, idx, name))
            for meth, col in LABEL_BY_METHOD.items():           # kích thước nhóm trên MỌI khách có snapshot (gồm No_Acquisition_Event)
                vc = lab[col].value_counts()
                sizes += [{'month_idx': idx, 'config': name, 'method': meth, 'label': str(k), 'n_khach': int(v), 'pct_observed': v / len(lab)}
                          for k, v in vc.items() if v > 0]
            if prev is not None and prev[0] == idx - 1:
                stab += stability_rows(prev[1][name], lab, idx, name)
            if write:
                RFM_LABEL_DIR.mkdir(parents=True, exist_ok=True)
                lab.to_parquet(RFM_LABEL_DIR / f'rfm_labels_{name}_{month_ts(idx):%Y-%m}.parquet', index=False)
        prev = (idx, labs)
        if idx in (fit_idx, last_idx):
            keep[idx] = (labs, df)
        print(f'{month_ts(idx):%Y-%m}: {len(df):,} khách có snapshot | {len(labs)} cấu hình')
    fmt = lambda rows: pd.DataFrame(rows)
    return params, fmt(cov), fmt(sizes), fmt(stab), keep


In [13]:
def kmeans_scan_table(params):
    rows = [{'config': n, 'k_dung': c['kmeans']['k'], 'k_silhouette_max': c['kmeans']['k_best_silhouette'], 'n_fit': c['kmeans']['n_fit'],
             **{f'sil_k{k}': round(v['silhouette'], 3) for k, v in c['kmeans']['scan'].items()}}
            for n, c in params['configs'].items()]
    return pd.DataFrame(rows).set_index('config')


def quantile_tier_table(params):
    return pd.DataFrame([{'config': n, 'R_cuts': c['quantile']['R_cuts'], 'F_cuts': c['quantile']['F_cuts'],
                          'M_cuts': [round(x, 1) for x in c['quantile']['M_cuts']],
                          'so_tier_R_F_M': (len(c['quantile']['R_cuts']) + 1, len(c['quantile']['F_cuts']) + 1, len(c['quantile']['M_cuts']) + 1)}
                         for n, c in params['configs'].items()]).set_index('config')


SWAPS = [('Đổi F (M = số sản phẩm)', 'FE_MNP', 'FM_MNP'), ('Đổi F (M = thu nhập)', 'FE_MREN', 'FM_MREN'),
         ('Đổi M (F = F_events)', 'FE_MNP', 'FE_MREN'), ('Đổi M (F = F_event_months)', 'FM_MNP', 'FM_MREN')]


def swap_summary(keep_labs, df):
    rows, detail = [], {}
    for title, a, b in SWAPS:
        for col in ['q_segment', 'rb_segment_x_m', 'rb_segment']:
            rep = switch_report(keep_labs[a], keep_labs[b], df, col)
            rows.append({'thay đổi': title, 'so sánh': f'{a} → {b}', 'nhãn': col, 'n_chung': rep['n_common'],
                         'n_đổi_nhóm': rep['n_changed'], 'pct_đổi_nhóm': round(rep['pct_changed'], 4)})
            detail[(title, col)] = rep
    return pd.DataFrame(rows), detail


In [14]:
LABEL_IDXS = sorted({FIT_IDX, SELECT_IDX, *TEST_IDXS, LAST_IDX})
assert FIRST_IDX < FIT_IDX and LABEL_IDXS[-1] <= LAST_IDX, 'Tháng chốt nằm ngoài phạm vi dữ liệu'
assert WRITE_MONTHLY_FEATURES, 'Đợt 2 đọc bảng feature từ đĩa; đặt WRITE_MONTHLY_FEATURES=True ở mục 0'
rfm_params, cov, sizes, stab, keep = run_rfm_experiments(LABEL_IDXS, FIT_IDX, LAST_IDX, write=WRITE_RFM_LABELS)
params_to_json(rfm_params, Path(WORK_DIR) / 'rfm_thresholds.json')
print('Tham số đã học từ', rfm_params['meta']['fit_month'], '| khách fit:', f"{rfm_params['meta']['n_fit_customers']:,}",
      '| median income dùng để điền:', round(rfm_params['meta']['renta_fill_value']))
print('Ngưỡng quantile (số tier thực tế ở cột cuối; ít hơn 3 nghĩa là dữ liệu trùng giá trị nhiều):')
display(quantile_tier_table(rfm_params))
print('K-means: k dùng (k_dung), k có silhouette lớn nhất (k_silhouette_max) và silhouette theo k:')
display(kmeans_scan_table(rfm_params))
display(pd.DataFrame(rfm_params['configs'][PRIMARY_CONFIG]['kmeans']['centroids_orig']).round(2))

2015-08: 843,201 khách có snapshot | 6 cấu hình
2015-09: 865,440 khách có snapshot | 6 cấu hình
2016-01: 916,269 khách có snapshot | 6 cấu hình
2016-02: 920,904 khách có snapshot | 6 cấu hình
2016-03: 925,076 khách có snapshot | 6 cấu hình
2016-04: 928,274 khách có snapshot | 6 cấu hình
2016-05: 931,453 khách có snapshot | 6 cấu hình
Tham số đã học từ 2015-08-28 | khách fit: 843,201 | median income dùng để điền: 101733
Ngưỡng quantile (số tier thực tế ở cột cuối; ít hơn 3 nghĩa là dữ liệu trùng giá trị nhiều):


,R_cuts,F_cuts,M_cuts,so_tier_R_F_M
config,,,,
FE_MNP,"[1.0, 3.0]","[1.0, 2.0]","[2.0, 4.0]","(3, 3, 3)"
FE_MREN,"[1.0, 3.0]","[1.0, 2.0]","[83965.0, 141112.8]","(3, 3, 3)"
FM_MNP,"[1.0, 3.0]","[1.0, 2.0]","[2.0, 4.0]","(3, 3, 3)"
FM_MREN,"[1.0, 3.0]","[1.0, 2.0]","[83965.0, 141112.8]","(3, 3, 3)"
FE_MRENfill,"[1.0, 3.0]","[1.0, 2.0]","[91647.5, 128852.2]","(3, 3, 3)"
FM_MRENfill,"[1.0, 3.0]","[1.0, 2.0]","[91647.5, 128852.2]","(3, 3, 3)"


K-means: k dùng (k_dung), k có silhouette lớn nhất (k_silhouette_max) và silhouette theo k:


,k_dung,k_silhouette_max,n_fit,sil_k3,sil_k4,sil_k5,sil_k6,sil_k7,sil_k8
config,,,,,,,,,
FE_MNP,5,4,116554,0.325,0.346,0.337,0.338,0.298,0.334
FE_MREN,5,4,100083,0.295,0.303,0.280,0.280,0.284,0.293
FM_MNP,5,5,116554,0.336,0.378,0.388,0.332,0.387,0.347
FM_MREN,5,3,100083,0.340,0.332,0.333,0.321,0.334,0.318
FE_MRENfill,5,4,116554,0.289,0.313,0.289,0.282,0.279,0.292
FM_MRENfill,5,4,116554,0.346,0.352,0.337,0.317,0.345,0.333


,rank,R,F,M,n_fit
0,1,1.03,2.50,5.68,16801
1,2,1.51,2.22,2.34,27189
2,3,3.04,1.05,6.29,13672
3,4,0.97,1.00,2.15,29601
4,5,4.62,1.02,2.10,29291


> ### 📝 Chú thích ngưỡng quantile, silhouette và tâm cụm k-means
> **Ngưỡng quantile (học trên 08/2015):**
> - R cắt tại [1, 3]: R ≤ 1 là mới mở trong 1 tháng gần nhất, R ≥ 4 là lâu rồi. F cắt tại [1, 2] (F=1, F=2, F≥3).
> - M_n_products cắt tại [2, 4] (≤2, 3–4, >4 SP). M_renta cắt tại ~84 nghìn và ~141 nghìn.
> - Cả 4 cấu hình đều ra đủ 3 tier cho mỗi trục, không bị suy biến do trùng giá trị.
>
> **Silhouette theo k (mức 0,3–0,4 là trung bình, không phải cụm tách bạch):**
> - **FM_MNP cao nhất: 0,388 tại k=5** (cũng là cấu hình duy nhất có đỉnh trùng k=5). FE_MREN thấp nhất (~0,28–0,30).
> - Cấu hình dùng `F_event_months` luôn nhỉnh hơn `F_events`, và `M_n_products` nhỉnh hơn `M_renta` → hỗ trợ chọn FM_MNP.
> - Chênh lệch giữa các k nhỏ nên việc cố định k=5 là hợp lý, không chắc k=4 hay 6 tệ hơn.
>
> **Tâm cụm k-means (FM_MNP, đơn vị gốc):**
> - **KM1** (R≈1, F≈2,5, M≈5,7): vừa mở gần đây, mở nhiều lần, giữ nhiều SP → giá trị cao nhất.
> - **KM2** (R≈1,5, F≈2,2, M≈2,3): gần đây, F cao nhưng ít SP → dấu hiệu mở rồi đóng (flicker).
> - **KM3** (R≈3, F≈1, M≈6,3): giữ nhiều SP nhưng đã lâu không mở thêm, nhóm nhỏ nhất.
> - **KM4** (R≈1, F=1, M≈2,2): mới mở lần đầu, ít SP.
> - **KM5** (R≈4,6, F≈1, M≈2,1): lâu không có hoạt động, thấp mọi trục → giá trị thấp nhất.
> - Thứ hạng chỉ đáng tin ở hai đầu (KM1 và KM5); KM2–KM4 không xếp hơn kém được.
> - `n_fit` 116.554: số khách có event và có M tại 08/2015.

---
## 6. So sánh bốn cấu hình

### 6.1. Độ phủ và quy mô nhóm

`pct_scored` = khách được chấm / khách có snapshot. Cấu hình dùng `M_renta` gốc bị mất độ phủ đúng bằng tỷ lệ thiếu `renta`; hai cấu hình `*fill` là kiểm tra phụ.

In [15]:
cov['month'] = cov.month_idx.map(lambda i: f'{month_ts(i):%Y-%m}')
display(cov.pivot(index='month', columns='config', values='pct_scored').round(3))
fill_rate = 1 - cov[cov.config == 'FM_MREN'].set_index('month').pct_M_known
print('Tỷ lệ phải điền income (khách có snapshot nhưng thiếu renta) theo tháng:'); display(fill_rate.round(3).to_frame('pct_filled').T)

sizes['month'] = sizes.month_idx.map(lambda i: f'{month_ts(i):%Y-%m}')
last = f'{month_ts(LAST_IDX):%Y-%m}'
for meth in ['quantile', 'kmeans', 'rule']:
    t = sizes[(sizes.month == last) & (sizes.config == PRIMARY_CONFIG) & (sizes.method == meth)]
    print(f'\n{PRIMARY_CONFIG} | {meth} | {last} (pct trên MỌI khách có snapshot, gồm {NO_EVENT})')
    display(t.sort_values('label')[['label', 'n_khach', 'pct_observed']].round(4).set_index('label'))
scored_only = sizes[~sizes.label.str.contains(NO_EVENT + '|Unscored|Unknown|\\?', regex=True)]
top = (scored_only[scored_only.month == last].groupby(['config', 'method']).agg(n_nhom=('label', 'nunique'), nhom_lon_nhat=('pct_observed', 'max')).round(3))
print('\nSố nhóm được chấm đầy đủ và tỷ trọng nhóm lớn nhất (trên mọi khách có snapshot), mọi cấu hình:'); display(top.unstack('method'))

config,FE_MNP,FE_MREN,FE_MRENfill,FM_MNP,FM_MREN,FM_MRENfill
month,,,,,,
2015-08,0.138,0.119,0.138,0.138,0.119,0.138
2015-09,0.148,0.127,0.148,0.148,0.127,0.148
2016-01,0.186,0.149,0.186,0.186,0.149,0.186
2016-02,0.194,0.152,0.194,0.194,0.152,0.194
2016-03,0.201,0.156,0.201,0.201,0.156,0.201
2016-04,0.207,0.158,0.207,0.207,0.158,0.207
2016-05,0.213,0.161,0.213,0.213,0.161,0.213


Tỷ lệ phải điền income (khách có snapshot nhưng thiếu renta) theo tháng:


month,2015-08,2015-09,2016-01,2016-02,2016-03,2016-04,2016-05
pct_filled,0.171,0.18,0.23,0.235,0.239,0.243,0.246



FM_MNP | quantile | 2016-05 (pct trên MỌI khách có snapshot, gồm No_Acquisition_Event)


,n_khach,pct_observed
label,,
No_Acquisition_Event,732966,0.7869
R1F1M1,45284,0.0486
R1F1M2,11960,0.0128
R1F1M3,6596,0.0071
R1F2M1,14131,0.0152
R1F2M2,6785,0.0073
R1F2M3,4081,0.0044
R1F3M1,6999,0.0075
R1F3M2,4955,0.0053


...

,n_khach,pct_observed
label,,
KM1,33294,0.0357
KM2,51329,0.0551
KM3,8144,0.0087
KM4,16686,0.0179
KM5,89034,0.0956
No_Acquisition_Event,732966,0.7869



FM_MNP | rule | 2016-05 (pct trên MỌI khách có snapshot, gồm No_Acquisition_Event)


,n_khach,pct_observed
label,,
Active|M_high,10765,0.0116
Active|M_low,13448,0.0144
Active|M_mid,21269,0.0228
At-risk|M_high,9331,0.0100
At-risk|M_low,14349,0.0154
At-risk|M_mid,16839,0.0181
Champion|M_high,24602,0.0264
Champion|M_low,5472,0.0059
Champion|M_mid,18385,0.0197



Số nhóm được chấm đầy đủ và tỷ trọng nhóm lớn nhất (trên mọi khách có snapshot), mọi cấu hình:


n_nhom               nhom_lon_nhat                
method      kmeans quantile rule        kmeans quantile   rule
config                                                        
FE_MNP           5       27   12         0.098    0.045  0.031
FE_MREN          5       27   12         0.053    0.016  0.020
FE_MRENfill      5       27   12         0.103    0.025  0.024
FM_MNP           5       27   12         0.096    0.049  0.027
FM_MREN          5       27   12         0.075    0.018  0.020
FM_MRENfill      5       27   12         0.099    0.028  0.027

> ### 📝 Chú thích độ phủ và quy mô nhóm (mục 6.1)
> - **`pct_scored`:** chỉ 13,8% (08/2015) → 21,3% (05/2016) khách được chấm. Cấu hình `M_renta` thấp hơn (11,9% → 16,1%) vì mất đúng phần khách thiếu renta. FE_ và FM_ có cùng độ phủ vì chỉ khác định nghĩa F.
> - **Tỷ lệ phải điền income** tăng 17,1% → 24,6%: ngày càng nhiều khách (đặc biệt cohort mới) không có renta hợp lệ, nên cấu hình dựa trên renta ngày càng thiên lệch.
> - **Quy mô nhóm (05/2016):** `No_Acquisition_Event` chiếm **78,7%**. Quantile có 27 nhóm, nhóm lớn nhất chỉ 4,9% (R1F1M1, 45.284 khách), nhiều nhóm dưới 0,3% → quá vụn để hành động.
> - **K-means** chỉ 5 nhóm, lớn nhất KM5 (9,6%), dễ gán chiến lược hơn. Rule-based 12 nhóm, ít vụn hơn quantile.

### 6.2. Ổn định theo tháng

Tỷ lệ cùng khách (được chấm ở cả hai tháng liền kề) giữ nguyên nhãn. Nhãn đổi một phần vì R tự tăng theo thời gian, nên đây là chỉ số mô tả, không tự nó nói cấu hình nào tốt hơn.

In [16]:
stab['month'] = stab.month_idx.map(lambda i: f'{month_ts(i):%Y-%m}')
print('Trung bình các cặp tháng liền kề:')
display(stab.groupby(['config', 'method']).pct_same_label.mean().unstack('method').round(3))
display(stab[stab.config == PRIMARY_CONFIG].pivot(index='month', columns='method', values='pct_same_label').round(3))

Trung bình các cặp tháng liền kề:


method,kmeans,quantile,rule
config,,,
FE_MNP,0.837,0.640,0.712
FE_MREN,0.853,0.694,0.796
FE_MRENfill,0.848,0.684,0.796
FM_MNP,0.814,0.637,0.711
FM_MREN,0.877,0.688,0.792
FM_MRENfill,0.860,0.678,0.793


method,kmeans,quantile,rule
month,,,
2015-09,0.762,0.576,0.673
2016-02,0.794,0.584,0.681
2016-03,0.829,0.688,0.736
2016-04,0.846,0.639,0.724
2016-05,0.839,0.697,0.739


> ### 📝 Chú thích độ ổn định theo tháng (mục 6.2)
> - **Trung bình tỷ lệ khách giữ nguyên nhãn giữa hai tháng liền kề:** k-means 0,81–0,88 > rule 0,71–0,80 > quantile 0,64–0,69.
> - K-means ổn định nhất vì chỉ có 5 nhóm; quantile có 27 ô nên chỉ cần R nhích 1 bậc là đổi ô.
> - FM_MREN cao nhất (0,877) vì `M_renta` hầu như không đổi theo tháng, chứ không hẳn vì chia cụm tốt hơn. FM_MNP (0,814) thấp hơn vì M_n_products biến động theo hành vi, đó lại là tín hiệu hữu ích.
> - Theo tháng của FM_MNP: tăng dần từ 0,76 (09/2015) lên ~0,84 (05/2016) khi dữ liệu ổn định hơn.
> - Lưu ý: đây là chỉ số mô tả (R tự tăng nên nhãn đổi một phần), không tự nó chứng minh cấu hình nào tốt hơn.

### 6.3. Khách nào đổi nhóm khi thay F hoặc M

So trên **cùng tập khách được chấm ở cả hai cấu hình** (tháng cuối). Chỉ dùng nhãn quantile và rule-based: cụm k-means được fit lại riêng cho từng cấu hình nên tên cụm giữa các cấu hình không tương đương. `rb_segment` chỉ dùng R/F nên đổi M phải cho 0% đổi nhóm (kiểm tra chéo).

In [17]:
swap_df, swap_detail = swap_summary(keep[LAST_IDX][0], keep[LAST_IDX][1])
display(swap_df)
title = 'Đổi F (M = số sản phẩm)'
rep = swap_detail[(title, 'q_segment')]
print(f'{title}: {rep["n_changed"]:,}/{rep["n_common"]:,} khách đổi nhóm quantile. Hồ sơ khách đổi (True) so với giữ nguyên (False):')
display(rep['profile'])
print('Luồng đổi nhóm lớn nhất:'); display(rep['top_flows'])
title = 'Đổi M (F = F_event_months)'
rep = swap_detail[(title, 'rb_segment_x_m')]
print(f'{title}: {rep["n_changed"]:,}/{rep["n_common"]:,} khách đổi nhóm rule-based (RF × M). Hồ sơ:')
display(rep['profile']); display(rep['top_flows'])

,thay đổi,so sánh,nhãn,n_chung,n_đổi_nhóm,pct_đổi_nhóm
0,Đổi F (M = số sản phẩm),FE_MNP → FM_MNP,q_segment,198487,25470,0.1283
1,Đổi F (M = số sản phẩm),FE_MNP → FM_MNP,rb_segment_x_m,198487,8676,0.0437
2,Đổi F (M = số sản phẩm),FE_MNP → FM_MNP,rb_segment,198487,8676,0.0437
3,Đổi F (M = thu nhập),FE_MREN → FM_MREN,q_segment,149775,18654,0.1245
4,Đổi F (M = thu nhập),FE_MREN → FM_MREN,rb_segment_x_m,149775,5475,0.0366
5,Đổi F (M = thu nhập),FE_MREN → FM_MREN,rb_segment,149775,5475,0.0366
6,Đổi M (F = F_events),FE_MNP → FE_MREN,q_segment,149775,93156,0.6220
7,Đổi M (F = F_events),FE_MNP → FE_MREN,rb_segment_x_m,149775,93857,0.6267
8,Đổi M (F = F_events),FE_MNP → FE_MREN,rb_segment,149775,0,0.0000
9,Đổi M (F = F_event_months),FM_MNP → FM_MREN,q_segment,149775,93156,0.6220


Đổi F (M = số sản phẩm): 25,470/198,487 khách đổi nhóm quantile. Hồ sơ khách đổi (True) so với giữ nguyên (False):


,R,F_events,F_event_months,F_gap,M_n_products,M_renta
changed,,,,,,
False,4.86,2.75,2.32,0.43,2.73,140173.437500
True,5.28,3.13,1.57,1.56,3.48,136910.890625


Luồng đổi nhóm lớn nhất:


,q_segment_a,q_segment_b,n_khach
0,R1F3M1,R1F2M1,3141
1,R1F3M2,R1F2M2,2491
2,R3F3M2,R3F2M2,1965
3,R1F3M3,R1F2M3,1707
4,R1F2M1,R1F1M1,1673
5,R2F3M2,R2F2M2,1500
6,R1F2M3,R1F1M3,1422
7,R1F2M2,R1F1M2,1410
8,R1F3M1,R1F1M1,1302
9,R2F3M3,R2F2M3,1235


Đổi M (F = F_event_months): 93,857/149,775 khách đổi nhóm rule-based (RF × M). Hồ sơ:


,R,F_events,F_event_months,F_gap,M_n_products,M_renta
changed,,,,,,
False,5.24,2.89,2.33,0.56,3.12,148048.34375
True,5.21,2.99,2.35,0.64,2.97,134833.31250


,rb_segment_x_m_a,rb_segment_x_m_b,n_khach
0,Lost|M_mid,Lost|M_low,7433
1,Lost|M_mid,Lost|M_high,7057
2,Champion|M_high,Champion|M_mid,6947
3,Lost|M_low,Lost|M_mid,6278
4,Champion|M_high,Champion|M_low,6014
5,Champion|M_mid,Champion|M_low,5313
6,Lost|M_low,Lost|M_high,4880
7,Champion|M_mid,Champion|M_high,4558
8,At-risk|M_mid,At-risk|M_low,4492
9,Active|M_mid,Active|M_low,4245


> ### 📝 Chú thích khách đổi nhóm khi thay F hoặc M (mục 6.3)
> - **Đổi F (F_events → F_event_months, M giữ nguyên):** chỉ 12,8% khách đổi nhóm quantile và 4,4% đổi nhóm rule-based. Hai cách đo F tương đối tương thích.
> - Khách đổi nhóm có **F_gap = F_events − F_event_months = 1,56** (so với 0,43 ở nhóm không đổi): họ mở nhiều SP trong cùng tháng hoặc mở lại SP, nên F_events cao hơn thực chất. Luồng lớn nhất là R1F3 → R1F2 (hạ F xuống). Đây là lý do F_event_months là lựa chọn chặt chẽ hơn.
> - **Đổi M (số SP → thu nhập):** 62% khách đổi nhóm. Hai trục M đo hai thứ rất khác nhau, **không thay thế được cho nhau**.
> - Luồng đổi M điển hình: `Lost|M_mid` ↔ `Lost|M_low/high`, `Champion|M_high` → `Champion|M_mid/low`: khách giữ nhiều SP chưa chắc thu nhập cao và ngược lại.
> - Kiểm tra chéo: `rb_segment` (chỉ dùng R/F) đổi M thì 0% đổi nhóm → code đúng.

### 6.4. Khách không có event, độ trôi theo tháng, hai trục M, flicker

**Nhóm không có event** (R = NaN) là phần lớn khách nhưng không được gán tier; cần biết họ là ai. **Độ trôi:** ngưỡng đóng băng ở 08/2015 nên khách không có event mới chỉ có R tăng, khối lượng dồn về nhóm cũ; tên nhóm mang nghĩa theo quần thể 08/2015. **Hai trục M** đo hai thứ khác nhau nếu tương quan thấp. **Flicker:** khách F cao nhưng đang giữ ít sản phẩm buộc phải đóng rồi mở lại; đo bao nhiêu event là mở lại cùng một sản phẩm.

In [18]:
df_last = keep[LAST_IDX][1]
prof = no_event_profile(df_last, window)
print('Quy mô (mọi khách có snapshot ở tháng cuối):'); display(prof['quy_mo'])
print('Theo loại đến (arrival_type):'); display(prof['arrival_type'])
print('Theo số sản phẩm:'); display(prof['M_band'])
print('Chỉ số khác:'); display(prof['chi_so'])

drift = sizes[(sizes.config == PRIMARY_CONFIG) & (sizes.method == 'kmeans')].pivot(index='month', columns='label', values='pct_observed').round(3)
print(f'Độ trôi k-means ({PRIMARY_CONFIG}), tỷ trọng trên mọi khách có snapshot:'); display(drift)
rb = sizes[(sizes.config == PRIMARY_CONFIG) & (sizes.method == 'rule')].assign(seg=lambda t: t.label.str.split('|').str[0])
print('Độ trôi rule-based (RF, gộp M):'); display(rb.pivot_table(index='month', columns='seg', values='pct_observed', aggfunc='sum').round(3))

print('Tương quan hạng giữa M_n_products và M_renta:'); display(m_correlation(df_last))
print('Kiểm flicker (F_event_months >= 3): F cao & M <= 1 sản phẩm so với F cao & M >= 2:'); display(flicker_check(events, df_last))

Quy mô (mọi khách có snapshot ở tháng cuối):


,n_khach,pct
nhom,,
Có event,198487,0.2131
No_Acquisition_Event,732966,0.7869


Theo loại đến (arrival_type):


arrival_type,legacy_from_start,new_in_window,registered_earlier_seen_later
nhom,,,
Có event,0.741,0.222,0.037
No_Acquisition_Event,0.631,0.139,0.230


Theo số sản phẩm:


...

seg,Active,At-risk,Champion,Lost,No_Acquisition_Event
month,,,,,
2015-08,0.086,0.037,0.016,NaN,0.862
2015-09,0.083,0.034,0.021,0.009,0.852
2016-01,0.068,0.039,0.038,0.041,0.814
2016-02,0.062,0.043,0.045,0.044,0.806
2016-03,0.059,0.043,0.048,0.051,0.799
2016-04,0.052,0.047,0.049,0.059,0.793
2016-05,0.049,0.044,0.052,0.069,0.787


Tương quan hạng giữa M_n_products và M_renta:


,n,spearman
tập,,
mọi khách có snapshot,702435,0.076
khách có event,149775,0.149


Kiểm flicker (F_event_months >= 3): F cao & M <= 1 sản phẩm so với F cao & M >= 2:


,n_khach,event_tb,sp_khac_nhau_tb,pct_khach_co_mo_lai,repeat_share
nhom,,,,,
"F cao, M thấp",8818,4.40,2.20,0.894,0.501
"F cao, M cao",55350,5.36,2.75,0.882,0.487


> ### 📝 Chú thích nhóm không có event, độ trôi, hai trục M và flicker (mục 6.4)
> **Nhóm No_Acquisition_Event (78,7%, 732.966 khách):**
> - Phần lớn giữ ít SP: 86% có ≤1 SP (so với 29% ở nhóm có event). Số SP trung bình 0,93 so với 2,83.
> - Chỉ 29,7% đang active (nhóm có event: 89,6%). Đây là nhóm ít gắn bó, cần chiến lược kích hoạt riêng, đừng coi là "khách giá trị thấp" hay "đã churn".
> - Tenure trung bình gần nhau (13,4 so với 14,0 tháng) → chưa mở event không phải do khách mới. Nhóm `registered_earlier_seen_later` chiếm 23% ở đây (so với 3,7% ở nhóm có event), nghĩa là cohort 07/2015 chủ yếu chưa có event.
>
> **Độ trôi theo tháng (ngưỡng đóng băng ở 08/2015):**
> - K-means: KM5 tăng 3,5% → 9,6%, KM1/KM2 cũng tăng nhẹ, KM3/KM4 giảm. Khi thêm khách có event, họ dồn vào nhóm "lâu không mở" vì R tự tăng.
> - Rule-based: Lost từ 0 → 6,9%, Active giảm 8,6% → 4,9%. Tên nhóm chỉ có nghĩa theo quần thể 08/2015, cần thận trọng khi đọc về sau.
>
> **Tương quan hạng M_n_products và M_renta:** chỉ 0,08 (mọi khách) và 0,15 (khách có event) → gần như độc lập, giữ cả hai trục M là có lý do.
>
> **Flicker (F_event_months ≥ 3):** nhóm F cao đều có ~88–89% khách mở lại SP, `repeat_share` ~0,49–0,50 (khoảng một nửa event là mở lại cùng SP). Nhóm F cao nhưng M thấp có 4,4 event trung bình trên 2,2 SP khác nhau → mở đóng liên tục. F cao không đồng nghĩa khách tốt nếu không xét kèm M.

---
## 7. Kiểm chứng dự báo: nhóm nào hay mở sản phẩm mới? (lift@10% và PR-AUC)

**Câu hỏi:** tại tháng chốt *t*, nhóm RFM của khách có dự báo được việc khách **mở ≥ 1 sản phẩm mới trong 3 tháng sau** (*t*+1 … *t*+3) không?

**Cách làm:**
- **Nhãn y** = 1 nếu khách có event trong *t*+1 … *t*+3. Chỉ event sau tháng chốt được dùng làm nhãn.
- **Điểm của khách** = tỷ lệ mở sản phẩm của nhóm đó ở **tháng học (08/2015)**, làm mượt cho nhóm nhỏ. Nhóm chưa từng thấy nhận tỷ lệ chung.
- **Đánh giá** trên các tháng sau: lift@10% (xử lý đồng điểm theo kỳ vọng) và PR-AUC (`pr_auc_x_base` = PR-AUC chia cho tỷ lệ mở chung, 1,0 là ngẫu nhiên).
- **Hai phạm vi:** *mọi khách* (gồm nhóm `No_Acquisition_Event`) và *khách có event* (xem riêng các nhóm RFM có phân biệt được hành vi không).
- **Mốc so sánh:** chỉ tách "có / không có event". Nhóm RFM chỉ có giá trị thêm nếu vượt được mốc này.

**Lưu ý thời gian:**
- Tháng 03 và 04/2016 bị bỏ vì chưa đủ 3 tháng sau trong dữ liệu (dữ liệu kết thúc 05/2016).
- Cột `strict` = True khi tháng đánh giá ≥ tháng học + 3 tháng (nhãn của tháng học đã biết trước tháng đó). 09/2015 có `strict=False` vì cửa sổ nhãn chồng lấn với tháng học, chỉ để tham khảo.
- Khách vắng mặt sau tháng chốt không thể có event, nên tỷ lệ mở bị thấp nhẹ ở mọi nhóm (censoring); không xử lý riêng ở đây.

In [19]:
from sklearn.metrics import average_precision_score

HORIZON = 3          # nhãn: khách có mở sản phẩm mới trong 3 tháng SAU tháng chốt
LIFT_FRAC = 0.10     # lift@10%
SMOOTH_N = 20        # làm mượt tỷ lệ của nhóm nhỏ về tỷ lệ chung (số khách "ảo")


def load_labels(config, idx):
    p = RFM_LABEL_DIR / f'rfm_labels_{config}_{month_ts(idx):%Y-%m}.parquet'
    assert p.exists(), f'Thiếu {p}. Chạy mục 5 với WRITE_RFM_LABELS=True trước.'
    return pd.read_parquet(p, columns=['customer_id', *LABEL_BY_METHOD.values()])


def future_open(events, ids, t_idx, horizon=HORIZON):
    """1 nếu khách có >= 1 event trong các tháng t+1 .. t+horizon. Chỉ event SAU tháng chốt được dùng làm nhãn."""
    ev = events[(events.month_idx > t_idx) & (events.month_idx <= t_idx + horizon)]
    return pd.Index(ids).isin(ev.customer_id.unique()).astype('int8')


def segment_rates(seg, y, smooth=SMOOTH_N):
    """Tỷ lệ mở SP của từng nhóm ở tập học, làm mượt về tỷ lệ chung để nhóm nhỏ không nhiễu."""
    d = pd.DataFrame({'seg': np.asarray(seg, dtype=str), 'y': np.asarray(y)})
    base = float(d.y.mean())
    g = d.groupby('seg').y.agg(['sum', 'size'])
    return (g['sum'] + smooth * base) / (g['size'] + smooth), base


def lift_at(y, score, frac=LIFT_FRAC):
    """Lift của top `frac` khách theo điểm. Khách cùng nhóm có cùng điểm → nhóm cuối được lấy một phần theo tỷ lệ
    (kỳ vọng khi bốc ngẫu nhiên trong nhóm), nên kết quả không phụ thuộc thứ tự dòng."""
    d = pd.DataFrame({'y': np.asarray(y), 's': np.asarray(score)})
    base = d.y.mean()
    if base == 0:
        return np.nan
    g = d.groupby('s').y.agg(pos='sum', n='size').sort_index(ascending=False)
    k, got, pos = frac * len(d), 0.0, 0.0
    for r in g.itertuples():
        take = min(r.n, k - got)
        if take <= 0:
            break
        pos += r.pos * take / r.n
        got += take
    return (pos / k) / base


def evaluate_segments(train_seg, y_train, eval_seg, y_eval):
    """Điểm của khách = tỷ lệ mở SP của nhóm đó ở tập học (nhóm chưa từng thấy → tỷ lệ chung). Tính trên tập đánh giá."""
    rates, base_tr = segment_rates(train_seg, y_train)
    score = pd.Series(np.asarray(eval_seg, dtype=str)).map(rates).fillna(base_tr).to_numpy()
    base = float(np.mean(y_eval))
    ap = average_precision_score(y_eval, score)
    return {'n': int(len(y_eval)), 'base_rate': round(base, 4), 'pr_auc': round(ap, 4),
            'pr_auc_x_base': round(ap / base, 2), 'lift10': round(lift_at(y_eval, score), 2)}


def segment_table(seg, y):
    """Tỷ lệ mở SP thực tế của từng nhóm trên tập đánh giá."""
    d = pd.DataFrame({'nhom': np.asarray(seg, dtype=str), 'y': np.asarray(y)})
    t = d.groupby('nhom').y.agg(n_khach='size', ty_le_mo='mean')
    t['lift_nhom'] = t.ty_le_mo / d.y.mean()
    return t.sort_values('ty_le_mo', ascending=False).round(4)

In [20]:
EVAL_IDXS = [i for i in (SELECT_IDX, *TEST_IDXS) if i + HORIZON <= LAST_IDX]
SKIPPED = [i for i in (SELECT_IDX, *TEST_IDXS) if i + HORIZON > LAST_IDX]
fmt_m = lambda idxs: [f'{month_ts(i):%Y-%m}' for i in idxs]
print('Đánh giá ở các tháng:', fmt_m(EVAL_IDXS), '| bỏ vì chưa đủ', HORIZON, 'tháng sau trong dữ liệu:', fmt_m(SKIPPED))

# Tháng học (08/2015): tỷ lệ mở SP của từng nhóm
train_labs = {n: load_labels(n, FIT_IDX) for n in ALL_CONFIGS}
y_train = {n: future_open(events, lab.customer_id.to_numpy(), FIT_IDX) for n, lab in train_labs.items()}
no_ev_tr = (train_labs[PRIMARY_CONFIG].q_segment.astype(str) == NO_EVENT).to_numpy()
base_seg_train = np.where(no_ev_tr, 'no_event', 'has_event')

rows = []
for idx in EVAL_IDXS:
    # strict = nhãn của tháng học (08/2015 + 3 tháng) đã biết TRƯỚC tháng đánh giá → không rò rỉ nhãn
    strict = idx >= FIT_IDX + HORIZON
    mname = f'{month_ts(idx):%Y-%m}'
    for name in ALL_CONFIGS:
        lab = load_labels(name, idx)
        y = future_open(events, lab.customer_id.to_numpy(), idx)
        has_ev = (lab.q_segment.astype(str) != NO_EVENT).to_numpy()
        for meth, col in LABEL_BY_METHOD.items():
            for scope, m in (('mọi khách', np.ones(len(lab), bool)), ('khách có event', has_ev)):
                rows.append({'month': mname, 'strict': strict, 'config': name, 'method': meth, 'scope': scope,
                             **evaluate_segments(train_labs[name][col], y_train[name], lab[col].to_numpy()[m], y[m])})
        if name == PRIMARY_CONFIG:    # mốc so sánh: chỉ tách "có / không có event", không chia nhóm nào khác
            rows.append({'month': mname, 'strict': strict, 'config': 'baseline', 'method': 'chi_tach_co_khong_event', 'scope': 'mọi khách',
                         **evaluate_segments(base_seg_train, y_train[name], np.where(has_ev, 'has_event', 'no_event'), y)})

val = pd.DataFrame(rows)
val.to_csv(Path(WORK_DIR) / 'rfm_validation_results.csv', index=False)

s = val[val.strict]
print('\nTóm tắt (trung bình các tháng strict) — pr_auc_x_base và lift10, càng cao càng tốt, 1,0 = ngẫu nhiên:')
for scope in ['mọi khách', 'khách có event']:
    print(f'\nPhạm vi: {scope}')
    display(s[s.scope == scope].groupby(['config', 'method'])[['base_rate', 'pr_auc_x_base', 'lift10']].mean().round(3))
print('\nChi tiết từng tháng, cấu hình chính và mốc so sánh (phạm vi: mọi khách):')
display(val[val.config.isin([PRIMARY_CONFIG, 'baseline']) & (val.scope == 'mọi khách')]
        .set_index(['month', 'strict', 'config', 'method'])[['n', 'base_rate', 'pr_auc', 'pr_auc_x_base', 'lift10']])

# Tỷ lệ mở SP thực tế của từng nhóm, cấu hình chính, tháng đánh giá cuối
last_eval = EVAL_IDXS[-1]
lab = load_labels(PRIMARY_CONFIG, last_eval)
y_last = future_open(events, lab.customer_id.to_numpy(), last_eval)
for meth, col in LABEL_BY_METHOD.items():
    print(f'\n{PRIMARY_CONFIG} | {meth} | {month_ts(last_eval):%Y-%m} — tỷ lệ mở SP trong {HORIZON} tháng sau (toàn bộ khách: {y_last.mean():.4f})')
    display(segment_table(lab[col], y_last))

Đánh giá ở các tháng: ['2015-09', '2016-01', '2016-02'] | bỏ vì chưa đủ 3 tháng sau trong dữ liệu: ['2016-03', '2016-04']

Tóm tắt (trung bình các tháng strict) — pr_auc_x_base và lift10, càng cao càng tốt, 1,0 = ngẫu nhiên:

Phạm vi: mọi khách


base_rate  pr_auc_x_base  lift10
config      method                                                   
FE_MNP      kmeans                       0.078          4.285   5.150
            quantile                     0.078          4.525   5.260
            rule                         0.078          4.510   4.915
FE_MREN     kmeans                       0.078          4.185   4.930
            quantile                     0.078          4.510   5.325
            rule                         0.078          4.380   4.910
FE_MRENfill kmeans                       0.078          4.245   5.065
            quantile                     0.078          4.510   5.320
            rule                         0.078          4.345   4.910
FM_MNP      kmeans                       0.078          4.040   5.165
            quantile                     0.078          4.540   5.155
            rule                         0.078          4.580   4.895
FM_MREN     kmeans                       0.078          4.055   5.035
            quantile                     0.078          4.505   5.095
            rule                         0.078          4.430   4.895
FM_MRENfill kmeans                       0.078          4.105   5.185
            quantile                     0.078          4.510   5.090
            rule                         0.078          4.395   4.895
baseline    chi_tach_co_khong_event      0.078          3.200   3.940


Phạm vi: khách có event


base_rate  pr_auc_x_base  lift10
config      method                                    
FE_MNP      kmeans        0.308          1.365   1.620
            quantile      0.308          1.445   1.690
            rule          0.308          1.390   1.685
FE_MREN     kmeans        0.308          1.330   1.590
            quantile      0.308          1.440   1.650
            rule          0.308          1.370   1.625
FE_MRENfill kmeans        0.308          1.350   1.580
            quantile      0.308          1.440   1.650
            rule          0.308          1.365   1.620
FM_MNP      kmeans        0.308          1.280   1.375
            quantile      0.308          1.450   1.760
            rule          0.308          1.410   1.785
FM_MREN     kmeans        0.308          1.290   1.450
            quantile      0.308          1.440   1.720
            rule          0.308          1.385   1.715
FM_MRENfill kmeans        0.308          1.305   1.455
            quantile      0.308          1.445   1.710
            rule          0.308          1.380   1.705


Chi tiết từng tháng, cấu hình chính và mốc so sánh (phạm vi: mọi khách):


...

,n_khach,ty_le_mo,lift_nhom
nhom,,,
R3F3M1,7635,0.5454,7.3142
R2F3M1,5109,0.5036,6.7541
R3F3M2,10865,0.4794,6.4297
R2F3M2,3984,0.4649,6.2343
R2F3M3,2974,0.4492,6.0246
R3F3M3,10658,0.4491,6.0236
R1F3M2,2702,0.3627,4.8642
R1F3M3,2238,0.3557,4.7700
R1F3M1,3308,0.3501,4.6947



FM_MNP | kmeans | 2016-02 — tỷ lệ mở SP trong 3 tháng sau (toàn bộ khách: 0.0746)


,n_khach,ty_le_mo,lift_nhom
nhom,,,
KM1,29653,0.4309,5.7791
KM2,47217,0.3984,5.3429
KM4,22511,0.2670,3.5805
KM3,9500,0.2340,3.1382
KM5,69777,0.1660,2.2266
No_Acquisition_Event,742246,0.0233,0.3119



FM_MNP | rule | 2016-02 — tỷ lệ mở SP trong 3 tháng sau (toàn bộ khách: 0.0746)


,n_khach,ty_le_mo,lift_nhom
nhom,,,
Champion|M_low,4463,0.5913,7.9301
Champion|M_mid,15775,0.4931,6.6125
Champion|M_high,20987,0.4529,6.0733
Active|M_high,13293,0.2954,3.9619
At-risk|M_high,9548,0.2844,3.8135
Active|M_low,18067,0.2712,3.6373
Active|M_mid,26009,0.2698,3.6177
Lost|M_high,10799,0.2154,2.8886
At-risk|M_mid,15160,0.2110,2.8300


> ### 📝 Cách đọc kết quả mục 7
> - **`base_rate`** là tỷ lệ khách mở sản phẩm trong 3 tháng sau. Mọi chỉ số khác nên đọc so với con số này.
> - **`lift10` > 1** và **`pr_auc_x_base` > 1**: nhóm có ích hơn bốc ngẫu nhiên. Càng cao càng tốt.
> - **So với dòng `baseline`:** nếu k-means hay quantile chỉ ngang `baseline` ở phạm vi *mọi khách*, nghĩa là phần lớn tín hiệu đến từ việc tách có/không có event, các nhóm RFM chưa thêm được gì.
> - **Phạm vi *khách có event***: đây mới là phép thử cho các nhóm RFM. Nếu lift ≈ 1, các nhóm KM1–KM5 chưa phân biệt được hành vi mở sản phẩm.
> - **Bảng tỷ lệ mở theo nhóm** cho thấy nhóm nào cao, nhóm nào thấp. Nếu thứ tự không khớp thứ hạng KM1 → KM5 thì tên "nhóm tốt/xấu" dựa trên R/F/M chưa trùng với khả năng mở sản phẩm.
> - **Chênh lệch nhỏ (khoảng vài phần trăm lift)** giữa các cấu hình có thể chỉ là nhiễu: chưa tính khoảng tin cậy, và chỉ có 2 tháng `strict`.


> ### 📝 Nhận xét kết quả mục 7 (kiểm chứng dự báo)
> **Đánh giá ở 3 tháng:** 09/2015 (`strict=False`, chỉ tham khảo), 01/2016 và 02/2016 (`strict=True`). Tháng 03–04/2016 bị bỏ vì chưa đủ 3 tháng sau. Tỷ lệ mở sản phẩm chung là **7,8%** (mọi khách) và **30,8%** (khách có event).
>
> **1. Nhóm RFM có ích, thắng mốc so sánh.**
> - Mốc chỉ tách "có / không có event": lift@10% = 3,94, PR-AUC/base = 3,2.
> - Mọi cấu hình × phương pháp đạt lift 4,9–5,3 và PR-AUC/base 4,0–4,6.
> - Phần lớn tín hiệu (khoảng 3,9 / 5) đến từ việc có event hay không; RFM thêm khoảng +1 lift.
> - Nhóm `No_Acquisition_Event` chỉ mở **2,3%** (lift 0,31), thấp hơn hẳn nhóm có event; ngay cả nhóm có event kém nhất (KM5, 16,6%) vẫn cao hơn khoảng 7 lần.
>
> **2. Trong nhóm khách có event, khả năng dự báo ở mức vừa.** Lift@10% chỉ 1,4–1,8 (tỷ lệ mở chung 30,8%). Nhưng độ chênh giữa các nhóm khá rõ: nhóm cao nhất mở khoảng 55%, nhóm thấp nhất (R1F1M1) khoảng 13%.
>
> **3. K-means (cấu hình chính FM_MNP) là phương pháp yếu nhất về dự báo.**
> - Khách có event: lift 1,375 so với quantile 1,76 và rule 1,785. PR-AUC thấp hơn ở mọi cấu hình, khoảng cách ổn định ở cả hai tháng strict.
> - Một phần do k-means chỉ có 5 nhóm (quantile 27, rule 12) nên độ phân giải thấp hơn; chưa phải so sánh cùng số nhóm.
> - Lift@10% trên *mọi khách* của ba phương pháp gần như ngang nhau (~5,1–5,3).
>
> **4. Thứ hạng KM1–KM5 không khớp thực tế.** Tỷ lệ mở (02/2016): KM1 0,43 > KM2 0,40 > KM4 0,27 > KM3 0,23 > KM5 0,17. KM2 gần bằng KM1, KM3 và KM4 bị đảo. Đặt tên persona theo thứ hạng R/F/M cũ cần xem lại.
>
> **5. Vai trò của từng biến với nhãn mọi event, gồm mở lại (đọc từ bảng quantile; R3, F3, M3 là mức cao nhất). Với nhãn sản phẩm mới, xem nhận xét 7.2:**
> - **F mạnh nhất:** F3 mở khoảng 45–55%, F1 khoảng 13–30%.
> - **R đúng chiều:** gần đây thì mở nhiều hơn (R3F3 ≈ 0,45–0,55 so với R1F3 ≈ 0,35).
> - **M ngược chiều ở khách có event:** M thấp mở nhiều hơn (R3F3: M1 0,545 > M2 0,479 > M3 0,449; Champion|M_low 59% cao nhất). Hợp lý vì giữ ít sản phẩm thì còn nhiều chỗ để mở thêm. M đo giá trị, không đo khả năng mở thêm; vẫn cần cho CLV.
> - **M xuôi chiều ở nhóm không có event:** M_high mở 11,2%, M_mid 4,2%, M_low 1,8%. Nhóm `No_Acquisition_Event|M_high` (≈18,8 nghìn khách) nên xử lý riêng.
>
> **6. Các cấu hình gần như tương đương về dự báo.** `F_events` vs `F_event_months` và `M_n_products` vs `M_renta` chênh rất ít (quantile PR-AUC/base 1,44–1,45 ở khách có event). Giữ FM_MNP hợp lý vì độ bền và độ phủ, không phải vì dự báo. Rule-based vẫn ngang quantile (lift khách có event 1,785), và dễ giải thích hơn.
>
> **Giới hạn khi đọc:**
> - Chỉ có 2 tháng strict, chưa có khoảng tin cậy; chênh lệch nhỏ giữa các cấu hình chưa kết luận được.
> - Nhãn tính cả việc mở lại sản phẩm đã đóng, nên F cao dự báo tốt một phần do "hay mở lại thì sẽ mở lại". Chưa đo riêng việc mở sản phẩm *chưa từng giữ*.
> - 09/2015 thấp hơn (PR-AUC/base 3,4–3,7) vì cửa sổ nhãn chồng lấn; không dùng để so sánh.
> - Chưa đủ cơ sở để kết luận k-means là phương pháp chính; cần cân nhắc quantile hoặc rule-based.

### 7.2. Nhãn "sản phẩm mới thật sự" (không tính mở lại)

Nhãn ở 7.1 tính **mọi event 0→1**, gồm cả mở lại sản phẩm đã mở trước đó. Ở đây chỉ tính **lần mở đầu tiên** của từng cặp (khách, sản phẩm) trong bảng event.

- **Cách đọc:** so với kết quả 7.1 ở cùng phạm vi. Nếu lift và PR-AUC giảm nhiều, một phần khả năng dự báo ở 7.1 là "hay mở lại thì sẽ mở lại".
- **Giới hạn:** bảng event chỉ có các lần chuyển 0→1 từ 02/2015. Sản phẩm khách đã giữ từ snapshot đầu (01/2015), đóng rồi mở lại vẫn bị coi là "mới". Đây là xấp xỉ, không tuyệt đối.

In [21]:
# ---- 7.2. Nhãn "sản phẩm mới thật sự" (không tính mở lại) ----
# Lần mở đầu tiên của từng cặp (khách, sản phẩm) trong bảng event. Event nào không phải lần đầu là MỞ LẠI.
first_event = (events.groupby(['customer_id', 'product'], sort=False).month_idx.min()
               .rename('first_month').reset_index())
print(f'{len(events):,} event | {len(first_event):,} cặp khách-sản phẩm | '
      f'{1 - len(first_event) / len(events):.1%} event là MỞ LẠI sản phẩm đã mở trước đó trong cửa sổ quan sát')


def future_open_new(first_event, ids, t_idx, horizon=HORIZON):
    """1 nếu khách mở ít nhất một sản phẩm LẦN ĐẦU (trong cửa sổ quan sát) ở các tháng t+1 .. t+horizon."""
    fe = first_event[(first_event.first_month > t_idx) & (first_event.first_month <= t_idx + horizon)]
    return pd.Index(ids).isin(fe.customer_id.unique()).astype('int8')


def run_validation(y_fn, y_train_by_cfg, target_name):
    rows = []
    for idx in EVAL_IDXS:
        strict = idx >= FIT_IDX + HORIZON
        mname = f'{month_ts(idx):%Y-%m}'
        for name in ALL_CONFIGS:
            lab = load_labels(name, idx)
            y = y_fn(lab.customer_id.to_numpy(), idx)
            has_ev = (lab.q_segment.astype(str) != NO_EVENT).to_numpy()
            for meth, col in LABEL_BY_METHOD.items():
                for scope, m in (('mọi khách', np.ones(len(lab), bool)), ('khách có event', has_ev)):
                    rows.append({'target': target_name, 'month': mname, 'strict': strict, 'config': name, 'method': meth,
                                 'scope': scope, **evaluate_segments(train_labs[name][col], y_train_by_cfg[name],
                                                                     lab[col].to_numpy()[m], y[m])})
            if name == PRIMARY_CONFIG:
                rows.append({'target': target_name, 'month': mname, 'strict': strict, 'config': 'baseline',
                             'method': 'chi_tach_co_khong_event', 'scope': 'mọi khách',
                             **evaluate_segments(base_seg_train, y_train_by_cfg[name], np.where(has_ev, 'has_event', 'no_event'), y)})
    return pd.DataFrame(rows)


y_train_new = {n: future_open_new(first_event, lab.customer_id.to_numpy(), FIT_IDX) for n, lab in train_labs.items()}
val_new = run_validation(lambda ids, idx: future_open_new(first_event, ids, idx), y_train_new, 'sản phẩm mới')
val_new.to_csv(Path(WORK_DIR) / 'rfm_validation_results_new_product.csv', index=False)

s_new = val_new[val_new.strict]
for scope in ['mọi khách', 'khách có event']:
    print(f'\nNhãn "sản phẩm mới thật sự" | phạm vi: {scope} (trung bình các tháng strict)')
    display(s_new[s_new.scope == scope].groupby(['config', 'method'])[['base_rate', 'pr_auc_x_base', 'lift10']].mean().round(3))

# So sánh hai nhãn cho cấu hình chính (phạm vi khách có event): nhãn "mọi event" lấy từ `val` ở mục 7.1
both = pd.concat([val.assign(target='mọi event (gồm mở lại)'), val_new])
cmp_ = (both[(both.config == PRIMARY_CONFIG) & both.strict & (both.scope == 'khách có event')]
        .groupby(['target', 'method'])[['base_rate', 'pr_auc_x_base', 'lift10']].mean().round(3))
print('\nCấu hình chính, phạm vi khách có event: nhãn nào dễ dự báo hơn?')
display(cmp_)

last_eval = EVAL_IDXS[-1]
lab = load_labels(PRIMARY_CONFIG, last_eval)
y_last_new = future_open_new(first_event, lab.customer_id.to_numpy(), last_eval)
for meth, col in LABEL_BY_METHOD.items():
    print(f'\n{PRIMARY_CONFIG} | {meth} | {month_ts(last_eval):%Y-%m} — tỷ lệ mở SP MỚI trong {HORIZON} tháng sau (toàn bộ khách: {y_last_new.mean():.4f})')
    display(segment_table(lab[col], y_last_new))

561,710 event | 374,007 cặp khách-sản phẩm | 33.4% event là MỞ LẠI sản phẩm đã mở trước đó trong cửa sổ quan sát

Nhãn "sản phẩm mới thật sự" | phạm vi: mọi khách (trung bình các tháng strict)


base_rate  pr_auc_x_base  lift10
config      method                                                   
FE_MNP      kmeans                       0.039          1.995   3.020
            quantile                     0.039          2.210   3.150
            rule                         0.039          2.275   3.235
FE_MREN     kmeans                       0.039          1.855   2.765
            quantile                     0.039          2.130   3.040
            rule                         0.039          2.045   3.000
FE_MRENfill kmeans                       0.039          2.135   3.045
            quantile                     0.039          2.310   3.030
            rule                         0.039          2.220   3.050
FM_MNP      kmeans                       0.039          2.190   3.070
            quantile                     0.039          2.310   3.225
            rule                         0.039          2.295   3.335
FM_MREN     kmeans                       0.039          1.960   2.985
            quantile                     0.039          2.280   3.080
            rule                         0.039          2.060   3.070
FM_MRENfill kmeans                       0.039          2.185   3.030
            quantile                     0.039          2.410   3.060
            rule                         0.039          2.130   3.065
baseline    chi_tach_co_khong_event      0.039          1.785   2.590


Nhãn "sản phẩm mới thật sự" | phạm vi: khách có event (trung bình các tháng strict)


base_rate  pr_auc_x_base  lift10
config      method                                    
FE_MNP      kmeans          0.1          1.170   1.285
            quantile        0.1          1.340   1.805
            rule            0.1          1.210   1.345
FE_MREN     kmeans          0.1          1.055   1.110
            quantile        0.1          1.275   1.535
            rule            0.1          1.140   1.040
FE_MRENfill kmeans          0.1          1.275   1.685
            quantile        0.1          1.410   1.775
            rule            0.1          1.285   1.670
FM_MNP      kmeans          0.1          1.320   1.800
            quantile        0.1          1.415   1.970
            rule            0.1          1.230   1.355
FM_MREN     kmeans          0.1          1.140   1.120
            quantile        0.1          1.390   1.975
            rule            0.1          1.155   1.070
FM_MRENfill kmeans          0.1          1.320   1.875
            quantile        0.1          1.495   1.975
            rule            0.1          1.220   1.125


Cấu hình chính, phạm vi khách có event: nhãn nào dễ dự báo hơn?


...

,n_khach,ty_le_mo,lift_nhom
nhom,,,
R3F1M1,10413,0.2249,6.0338
R3F1M2,3354,0.2033,5.4551
R3F1M3,1703,0.1550,4.1588
R3F2M2,4778,0.1463,3.9247
R2F1M3,1349,0.1282,3.4404
R3F2M1,6336,0.1203,3.2264
R3F2M3,2834,0.1168,3.1333
R2F2M3,1789,0.1062,2.8492
R2F1M2,3086,0.1050,2.8166



FM_MNP | kmeans | 2016-02 — tỷ lệ mở SP MỚI trong 3 tháng sau (toàn bộ khách: 0.0373)


,n_khach,ty_le_mo,lift_nhom
nhom,,,
KM4,22511,0.1836,4.9255
KM3,9500,0.1036,2.7788
KM1,29653,0.0923,2.4753
KM2,47217,0.0864,2.3181
KM5,69777,0.0736,1.9739
No_Acquisition_Event,742246,0.0233,0.6238



FM_MNP | rule | 2016-02 — tỷ lệ mở SP MỚI trong 3 tháng sau (toàn bộ khách: 0.0373)


,n_khach,ty_le_mo,lift_nhom
nhom,,,
Active|M_mid,26009,0.1402,3.7618
Active|M_low,18067,0.1378,3.6959
Active|M_high,13293,0.1270,3.4067
No_Acquisition_Event|M_high,18795,0.1123,3.0117
Lost|M_high,10799,0.1009,2.7078
At-risk|M_high,9548,0.0969,2.5990
Champion|M_high,20987,0.0861,2.3111
Lost|M_mid,17432,0.0834,2.2377
Champion|M_mid,15775,0.0735,1.9710


> ### 📝 Nhận xét kết quả 7.2 (nhãn "sản phẩm mới thật sự")
> **Quy mô:** 561.710 event thuộc 374.007 cặp khách–sản phẩm, tức **33,4% event là mở lại** sản phẩm đã mở trước đó. Tỷ lệ mở sản phẩm mới trong 3 tháng sau là **3,9%** (mọi khách) và **10,0%** (khách có event), so với 7,8% và 30,8% của nhãn mọi event: phần lớn việc "mở sản phẩm" của khách có event là mở lại.
>
> **1. Nhóm RFM vẫn thắng mốc.** FM_MNP, mọi khách: lift 3,07–3,34 so với 2,59 của mốc; PR-AUC/base 2,19–2,31 so với 1,79.
>
> **2. Trong nhóm khách có event, dự báo yếu đến vừa.** FM_MNP: PR-AUC/base 1,23–1,42, lift 1,36 (rule) đến 1,97 (quantile). So với nhãn mọi event, quantile và k-means có lift cao hơn (1,97 và 1,80, so với 1,76 và 1,38), còn rule giảm (1,36 so với 1,79).
>
> **3. Nhóm "tốt nhất" theo RFM không phải nhóm mở sản phẩm mới nhiều nhất** (tỷ lệ mở thực tế, 02/2016):
> - **Quantile:** cao nhất là R3F1M1 (22,5%) và R3F1M2 (20,3%): gần đây, F thấp. Các nhóm F3 thấp hơn nhiều (R3F3M1 6,6%, R3F3M2 8,7%). Vậy **F đảo chiều**, R vẫn đúng chiều (cùng F thấp: R3 22,5% > R2 10,2% > R1 6,2%), M không nhất quán.
> - **K-means:** KM4 (vừa mở một lần, gần đây, ít sản phẩm) cao nhất, 18,4%. KM1, nhóm "tốt nhất" theo R/F/M, chỉ 9,2%; KM5 7,4%.
> - **Rule-based:** Active 12,7–14,0% cao hơn Champion 6,3–8,6%; `Champion|M_low` (6,3%) thấp hơn cả `Lost|M_high` (10,1%).
> - `No_Acquisition_Event|M_high` mở 11,2% (lift 3,0), cao hơn nhiều nhóm có event.
>
> **Cách hiểu (chưa kiểm định):** F cao chủ yếu là khách mở-đóng-mở lại, nên dự báo tốt việc mở lại (nhãn mọi event) nhưng không phải sản phẩm mới. Khách vừa mở lần đầu, còn gần đây (F thấp, R nhỏ) mới là nhóm đang mở thêm sản phẩm mới.
>
> **4. Hệ quả khi dùng:** nhóm RFM mô tả mức độ gắn bó và hoạt động, không đồng nghĩa với khả năng mua thêm sản phẩm mới. Để nhắm bán chéo, nên chọn nhóm theo nhãn mục tiêu (Active, R gần và F thấp), không mặc định chọn Champion hay KM1.
>
> **5. Các cấu hình điền median (`*fill`):** `FM_MRENfill` quantile có PR-AUC/base cao hơn FM_MNP khoảng 0,08–0,10 (lift ngang). Các cấu hình `M_renta` không điền thấp hơn một phần vì khoảng 25% khách thiếu renta không có nhóm M, nên so sánh `M_n_products` với `M_renta` cần tính đến độ phủ.

### 7.3. Bảng chốt để chọn cách chia nhóm

Bảng dưới chỉ giữ cấu hình chính `FM_MNP` và mốc so sánh đơn giản. Số liệu là **trung bình của hai tháng kiểm tra** 01 và 02/2016 (`strict=True`); mỗi nhãn và phạm vi có tỷ lệ nền riêng.

- **Lift@10%:** tỷ lệ mở sản phẩm trong 10% khách được xếp cao nhất, chia cho tỷ lệ mở chung của cùng phạm vi. `1` tương đương chọn ngẫu nhiên.
- **PR-AUC/base:** chất lượng xếp hạng toàn danh sách chia cho tỷ lệ mở chung. `1` tương đương xếp ngẫu nhiên.
- **Cách đọc:** trước hết so các phương pháp với `baseline` ở *mọi khách*, rồi xem riêng *khách có event* để biết các nhóm RFM có phân biệt thêm được không. Mốc `baseline` chỉ tách có/không có event nên không áp dụng cho phạm vi khách có event.

Hai tháng kiểm tra quá ít để khẳng định phương pháp nào luôn tốt nhất theo thời gian; các chênh lệch nhỏ chỉ nên xem là tham khảo.


In [22]:
# Một bảng gọn từ kết quả đã tính ở 7.1 và 7.2; không chạy bootstrap.
summary_73 = pd.concat([
    val.assign(target='mọi event'),
    val_new.assign(target='sản phẩm mới'),
], ignore_index=True)
summary_73 = summary_73[
    summary_73.strict & summary_73.config.isin([PRIMARY_CONFIG, 'baseline'])
].copy()
summary_73['cách chia'] = summary_73['method'].replace({
    'chi_tach_co_khong_event': 'baseline',
})
summary_73 = (summary_73
              .groupby(['target', 'scope', 'cách chia'], as_index=False)
              [['base_rate', 'lift10', 'pr_auc_x_base']].mean()
              .round(2))
display(summary_73.rename(columns={
    'target': 'nhãn dự báo', 'scope': 'phạm vi',
    'base_rate': 'tỷ lệ mở chung', 'lift10': 'Lift@10%',
    'pr_auc_x_base': 'PR-AUC/base',
}))


,nhãn dự báo,phạm vi,cách chia,tỷ lệ mở chung,Lift@10%,PR-AUC/base
0,mọi event,khách có event,kmeans,0.31,1.38,1.28
1,mọi event,khách có event,quantile,0.31,1.76,1.45
2,mọi event,khách có event,rule,0.31,1.78,1.41
3,mọi event,mọi khách,baseline,0.08,3.94,3.20
4,mọi event,mọi khách,kmeans,0.08,5.16,4.04
5,mọi event,mọi khách,quantile,0.08,5.16,4.54
6,mọi event,mọi khách,rule,0.08,4.89,4.58
7,sản phẩm mới,khách có event,kmeans,0.10,1.80,1.32
8,sản phẩm mới,khách có event,quantile,0.10,1.97,1.42
9,sản phẩm mới,khách có event,rule,0.10,1.36,1.23


**Kết luận từ bảng:**

1. Trên *mọi khách*, các cách chia RFM đều tốt hơn mốc chỉ tách có/không có event. Phần cải thiện nhỏ hơn khi dự báo *sản phẩm mới*.
2. Trong *khách có event*, **quantile** cho kết quả cân bằng nhất ở cả hai nhãn. Với sản phẩm mới, **rule-based** kém hơn; nhãn “Champion” không đồng nghĩa với khả năng mở sản phẩm mới cao nhất.
3. Chọn **`FM_MNP + quantile`** làm nhãn chính. Giữ rule-based để giải thích nhóm; dùng k-means để tham khảo. Kết luận này dựa trên hai tháng kiểm tra và ý nghĩa của các nhóm, không chỉ trên một con số Lift.


---
## 8. Vì sao chọn RFM như này

**Chốt:** dùng cấu hình **FM_MNP** = `R` × `F_event_months` × `M_n_products`. Phân nhóm bằng **quantile** (27 nhóm) làm nhãn chính; **rule-based** (12 nhóm) làm lớp diễn giải; **k-means** (5 cụm) giữ để tham khảo. Khách chưa từng có event tách riêng thành `No_Acquisition_Event`. Ngưỡng và tâm cụm học ở tháng 08/2015 rồi đóng băng.

### 8.1. Vì sao R, F, M dựng từ event mở sản phẩm
Dữ liệu Santander không có giao dịch tiền, chỉ có snapshot hằng tháng của 24 cờ sản phẩm. RFM vì vậy là *proxy* dựng từ **event mở sản phẩm** (cờ chuyển 0→1 giữa hai tháng liền kề):
- **R** = số tháng kể từ event gần nhất (càng nhỏ càng gần đây).
- **F** = số tháng có event (mức độ hay mở sản phẩm).
- **M** = số sản phẩm đang giữ (độ gắn bó với ngân hàng).

Cả ba chỉ dùng dữ liệu tới tháng chốt (mục 3.2: cắt dữ liệu tương lai thì bảng không đổi) và khớp 100% với notebook 01 (mục 3.1).

### 8.2. Vì sao tách riêng `No_Acquisition_Event`
- 78,7% khách chưa từng có event nên R không tồn tại (NaN). Điền 0 sẽ thành "vừa mua xong", điền số lớn sẽ thành "churn"; cả hai đều sai.
- Họ khác nhóm có event về bản chất: 86% giữ ≤ 1 sản phẩm, chỉ 29,7% đang active (nhóm có event: 89,6%), số sản phẩm trung bình 0,93 so với 2,83.
- Mục 7 xác nhận việc tách này có giá trị dự báo: nhóm này chỉ mở sản phẩm mới khoảng 2,3% trong 3 tháng sau (nhóm có event khoảng 30%), và riêng việc tách có/không có event đã cho lift@10% = 3,94.

### 8.3. Vì sao `F_event_months` thay vì `F_events`
- `F_events` đếm cả những lần mở lại sản phẩm vừa đóng: khoảng 88–89% khách F cao đã từng mở lại sản phẩm, và khoảng một nửa số event của nhóm này là mở lại cùng sản phẩm (trên toàn bộ, 33,4% event là mở lại sản phẩm đã mở trước đó).
- Nhóm khách đổi nhóm khi chuyển F có F_events trung bình 3,13 nhưng chỉ 1,57 tháng có event (chênh 1,56, so với 0,43 ở nhóm không đổi): họ chính là khách bị F_events thổi phồng.
- Đổi F không làm xáo trộn phân nhóm (12,8% khách đổi nhóm quantile, 4,4% với rule-based).
- **Dự báo (7.1–7.2):** với nhãn mọi event, hai cách F ngang nhau ở quantile và rule; riêng k-means thì F_events hơn (lift +0,20 đến +0,29) vì nó dự báo được việc mở lại. Với nhãn sản phẩm mới, `F_event_months` hơn ở quantile (lift hơn 0,14–0,20) và k-means (0,43–0,60), ngang ở rule. Vậy `F_event_months` không mất thông tin và tốt hơn khi mục tiêu là sản phẩm mới.

### 8.4. Vì sao `M_n_products` thay vì `M_renta`
- **Độ phủ:** 21,3% khách được chấm, so với 16,1% với M_renta, vì 24,6% khách thiếu renta. Điền median thì phủ được nhóm này, nhưng khoảng 25% khách bị dồn vào một giá trị thu nhập không có thật.
- **Ý nghĩa hành vi:** M_n_products tách rõ khách gắn bó và ít gắn bó. Nhóm có event có 29,7% khách giữ ≥ 4 sản phẩm, nhóm không event chỉ 2,2%.
- **Hai trục M đo hai thứ khác nhau:** tương quan hạng chỉ 0,08–0,15 và đổi M làm 62% khách đổi nhóm. M_renta ổn định theo tháng hơn (k-means 0,877 so với 0,814) vì thu nhập ít thay đổi, không phải vì phân cụm tốt hơn.
- **Dự báo (7.2):** so với M_renta không điền, M_n_products ngang hoặc tốt hơn ở hầu hết (hơn rõ ở k-means và rule với nhãn sản phẩm mới), nhưng phần lớn là do độ phủ: khoảng 25% khách thiếu renta không có nhóm M. Khi điền median, M_renta ngang M_n_products ở k-means và nhỉnh hơn nhẹ về PR-AUC ở quantile (1,50 so với 1,42 ở khách có event); chỉ rule vẫn kém hơn (lift 1,13 so với 1,36). Vì vậy dự báo không phân định rõ hai trục. M_n_products được chọn nhờ độ phủ không cần điền, ý nghĩa hành vi và việc rule-based dùng được.

### 8.5. Vì sao học ngưỡng ở 08/2015 rồi đóng băng
- Nếu ngưỡng và tâm cụm tính từ chính khách được đánh giá, chỉ số sẽ đẹp hơn thực tế. Chia theo thời gian (fit 08/2015, chọn 09/2015, kiểm thử 01–04/2016) mô phỏng đúng cách dùng thật.
- Đóng băng giữ cho nhãn có cùng ý nghĩa qua các tháng, nên so sánh được và theo dõi được một khách qua thời gian. Cái giá là hiện tượng trôi do R tự tăng (KM5 từ 3,5% lên 9,6%, Lost từ 0 lên 6,9%).

### 8.6. Vì sao quantile làm nhãn chính, rule-based diễn giải, k-means tham khảo
- **Mọi cấu hình vượt mốc** "chỉ tách có/không có event" ở cả hai nhãn và hai tháng (7.1–7.3): PR-AUC/base hơn +0,37 đến +1,42, lift hơn +0,47 đến +1,29.
- **Quantile ổn định nhất:** trong nhóm khách có event, quantile đứng đầu hoặc ngang đầu ở cả nhãn mọi event (lift 1,76) và nhãn sản phẩm mới (lift 1,85–2,09), không tụt ở kịch bản nào.
- **Rule-based:** ngang quantile ở nhãn mọi event, nhưng kém rõ ở nhãn sản phẩm mới trong nhóm khách có event (lift 1,35–1,36 so với 1,85–2,09), vì "Champion" dựa vào F ≥ 3 mà F cao phản ánh nhiều việc mở lại: ở nhãn sản phẩm mới, Champion chỉ mở 6,3–8,6% so với 12,7–14,0% của Active (7.2). Ở nhãn sản phẩm mới trên mọi khách, rule lại nhỉnh hơn, có thể nhờ chia nhóm `No_Acquisition_Event` theo M. Giữ làm lớp diễn giải (Champion / Active / At-risk / Lost).
- **K-means:** yếu nhất ở nhãn mọi event (khách có event: lift 1,37 so với 1,76 của quantile). Ở nhãn sản phẩm mới nó vượt rule trong nhóm khách có event nhưng vẫn kém quantile. Thứ hạng KM1–KM5 không khớp tỷ lệ mở thực tế (KM1 0,43 > KM2 0,40 > KM4 0,27 > KM3 0,23 > KM5 0,17), nên khó đặt tên persona theo thứ hạng.
- **Công bằng:** một phần k-means kém là do chỉ có 5 nhóm (quantile 27, rule 12).
- **Cái k-means hơn:** ổn định hơn (0,81–0,88 so với rule 0,71–0,80 và quantile 0,64–0,69) và gọn hơn, nên giữ làm cách nhìn tham khảo.

### 8.7. Biến nào quan trọng, và lưu ý khi dùng
- **Nhãn mọi event (gồm mở lại):** F mạnh nhất, R đúng chiều (gần đây thì mở nhiều hơn), M ngược chiều ở khách có event (giữ ít sản phẩm thì còn chỗ để mở thêm).
- **Nhãn sản phẩm mới:** R vẫn đúng chiều, nhưng **F đảo chiều** (F thấp mở sản phẩm mới nhiều hơn: R3F1 20–22%, R3F3 7–9%) và M không nhất quán. Nhóm mở sản phẩm mới nhiều nhất là khách mới mở, gần đây, F thấp (KM4, Active), không phải Champion hay KM1.
- M vẫn giữ vì là proxy giá trị cho CLV, không đổi chỉ vì dự báo.
- **Cách dùng:** nhóm RFM mô tả mức độ gắn bó và hoạt động. Để nhắm bán chéo sản phẩm mới, chọn nhóm theo nhãn mục tiêu (Active, R gần và F thấp), không mặc định chọn Champion.
- Nhóm RFM thêm khoảng +1 lift so với chỉ tách có/không có event ở nhãn mọi event, ít hơn ở nhãn sản phẩm mới. Trong nhóm khách có event, khả năng dự báo ở mức vừa (lift 1,4–2,1).

**Hạn chế:**
- Chỉ có 2 tháng kiểm tra hợp lệ (01 và 02/2016), nên chưa đánh giá được độ ổn định qua thời gian; chênh lệch nhỏ giữa các phương pháp chỉ mang tính tham khảo.
- Nhãn "sản phẩm mới" là xấp xỉ: sản phẩm khách đã giữ từ 01/2015 rồi đóng và mở lại vẫn bị tính là mới.
- 79% khách `No_Acquisition_Event` chưa có chiến lược riêng; nhóm `M_high` trong đó mở tới 11,2% (lift ≈ 1,5) nên xử lý riêng.

---
## 9. Mở rộng thêm

### 9.1. Tinh chỉnh ngưỡng rule-based bằng dữ liệu
Hiện `RULE_THRESHOLDS` (R ≤ 3 gần đây, R > 6 là Lost, F ≥ 3 là Champion) và mốc M (≤1 / 2–3 / ≥4 sản phẩm) được chọn sau EDA sơ bộ. Có thể kiểm tra kỹ hơn:
- Phân phối R, F, M ở tháng fit (khách có event) và kích thước các nhóm sau khi cắt.
- Tỷ lệ mở sản phẩm 3 tháng sau theo từng giá trị R, F, M (nhãn ở mục 7) để tìm chỗ tỷ lệ thay đổi rõ.
- So vài bộ ngưỡng ứng viên theo tiêu chí các nhóm tách rõ và đúng thứ tự Champion > Active > At-risk > Lost.
- Mốc M của rule-based (1 và 3) khác mốc quantile (2 và 4), có thể thử cả hai.
- Chỉ chọn ngưỡng bằng dữ liệu tháng fit (08/2015, nhãn 09–11/2015); dùng 01–02/2016 chỉ để kiểm tra lại, tránh rò rỉ.

### 9.2. Kết hợp quantile với chia nhóm `No_Acquisition_Event` theo M
Ở 7.3, rule-based dẫn đầu ở nhãn sản phẩm mới trên mọi khách, có thể nhờ chia nhóm không có event theo M (quantile chỉ có một nhóm). Có thể thử dùng quantile cho khách có event và chia `No_Acquisition_Event` thành M_low / M_mid / M_high, rồi so lại bằng cùng quy trình ở mục 7.
### 9.3. Kiểm tra độ chắc của kết quả mục 7.3
Bảng 7.3 chỉ tóm tắt hai tháng kiểm tra, nên chưa biết chênh lệch nhỏ giữa các cách chia nhóm có ổn định hay không. Nếu có thêm thời gian và dữ liệu, có thể:
- **Bootstrap theo khách:** lấy mẫu lại khách nhiều lần và tính lại Lift@10% cùng PR-AUC/base. So chênh lệch giữa quantile, k-means, rule-based và baseline trên cùng mẫu khách; báo cáo khoảng tin cậy thay vì chỉ một con số.
- **Kiểm tra theo thời gian:** đánh giá thêm nhiều tháng chốt độc lập để xem kết quả có giữ cùng chiều không. Bootstrap theo khách không thay thế được bước này.
- Ưu tiên cấu hình chính `FM_MNP`, hai nhãn “mọi event” và “sản phẩm mới”, cùng hai phạm vi “mọi khách” và “khách có event”. Chỉ mở rộng so sánh các cách đo F/M nếu kết quả chính chưa rõ.

Đây là hướng kiểm chứng tiếp theo, **chưa thực hiện trong notebook hiện tại**.
